# 0) Install & Login & Drive Mount

In [ ]:
!pip install -q wandb polars xgboost imblearn geonamescache pycountry

In [ ]:
import wandb
wandb.login()
# 계정 만들었으면, 2 누르고 본인 api 키 입력

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1) Imports

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,  # AUPRC/PR curve 핵심
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
import random

import joblib
import pickle
import os

import re
import pycountry
import geonamescache
from sklearn import set_config

# 2) 테이블 병합(Polars + Parquet)



In [ ]:
# 파일 경로 설정 (본인의 경로에 맞게 수정하세요)
base_path = "/content/drive/MyDrive/data"
parquet_dir = os.path.join(base_path, "parquet")
os.makedirs(parquet_dir, exist_ok=True)

trans_csv = os.path.join(base_path, "HI-Medium_Trans.csv")
acc_csv   = os.path.join(base_path, "HI-Medium_accounts.csv")
patterns_txt = os.path.join(base_path, "HI-Medium_Patterns.txt")

In [ ]:
# Parquet 저장 폴더
trans_parquet = os.path.join(parquet_dir, "HI-Medium_Trans.parquet")
acc_parquet   = os.path.join(parquet_dir, "HI-Medium_accounts.parquet")
master_parquet = os.path.join(parquet_dir, "HI-Medium_Master.parquet")

In [ ]:
# (처음 1회만) CSV -> Parquet 변환(스트리밍으로 처리되어 메모리 안전)
#    - 이미 parquet가 있으면 스킵
if not os.path.exists(trans_parquet):
    pl.scan_csv(
        trans_csv,
        infer_schema_length=10000,
        try_parse_dates=False
    ).sink_parquet(trans_parquet)

if not os.path.exists(acc_parquet):
    pl.scan_csv(
        acc_csv,
        infer_schema_length=10000,
    ).sink_parquet(acc_parquet)

In [ ]:
# Lazy 모드로 CSV 연결
q_trans = pl.scan_parquet(trans_parquet)
q_acc   = pl.scan_parquet(acc_parquet)

In [ ]:
#   Rates LazyFrame
#   - Currency_Rate.csv를 따로 scan_csv해도 되는데,
#     지금은 고정 테이블로 만든다고 했으니 lazy DF로 구성
# -----------------------------
rates_df = pl.DataFrame({
    "currency": [
        "US Dollar","Euro","UK Pound","Swiss Franc","Canadian Dollar",
        "Australian Dollar","Bitcoin","Saudi Riyal","Shekel","Brazil Real",
        "Yuan","Mexican Peso","Ruble","Rupee","Yen"
    ],
    "rate": [
        1.0,0.99,1.15,1.03,0.76,
        0.67,20000.0,0.266,0.29,0.19,
        0.14,0.05,0.017,0.0125,0.007
    ]
}).lazy()

In [ ]:
# -----------------------------
#  컬럼명 normalize (데이터마다 Account/Account.1 케이스가 있어서 방어)
# -----------------------------
# trans의 실제 컬럼을 먼저 확인하고 싶으면:
# print(q_trans.columns)  # Lazy에서도 가능

q_trans = (
    q_trans
    # 만약 컬럼명이 'Account', 'Account_duplicated_0'이라고 가정하면:
    .rename({
        "Account": "From Account",      # 첫 번째 Account는 보내는 사람
        "Account_duplicated_0": "To Account"       # 두 번째(중복) Account는 받는 사람
        # ※ 주의: 위에서 print(q_trans.columns)로 확인한 실제 이름과 다르면
        #         여기를 실제 이름에 맞춰 수정해야 합니다!
    })
)

# 이제 trans는 최소한 From Account / To Account 컬럼이 있다고 가정하고 진행

In [ ]:
# -----------------------------
#  Master build
#   1) Receiving Currency로 Amount Received USD
#   2) Payment Currency로 Amount Paid USD
#   3) accounts를 sender/receiver로 2번 left join (suffix 처리)
# -----------------------------
q_master = (
    q_trans

    # --- (1) Amount Received USD (Receiving Currency 기준)
    .join(
        rates_df.rename({"currency": "Receiving Currency", "rate": "recv_rate"}),
        on="Receiving Currency",
        how="left"
    )
    .with_columns([
        (pl.col("recv_rate").fill_null(1.0)).alias("recv_rate"),
        (pl.col("Amount Received") * pl.col("recv_rate")).alias("Amount_Received_USD")
    ])

    # --- (2) Amount Paid USD (Payment Currency 기준)
    .join(
        rates_df.rename({"currency": "Payment Currency", "rate": "pay_rate"}),
        on="Payment Currency",
        how="left"
    )
    .with_columns([
        (pl.col("pay_rate").fill_null(1.0)).alias("pay_rate"),
        (pl.col("Amount Paid") * pl.col("pay_rate")).alias("Amount_Paid_USD")
    ])

    # --- (3) Sender account join
    .join(
        q_acc.select(["Account Number", "Entity Name", "Bank Name"]),
        left_on="From Account",
        right_on="Account Number",
        how="left",
        suffix="_sender"
    )
    .rename({
        "Entity Name": "Sender_Entity",
        "Bank Name": "Sender_Bank_Name"
    })

    # --- (4) Receiver account join (컬럼 충돌 방지 위해 suffix 사용)
    .join(
        q_acc.select(["Account Number", "Entity Name", "Bank Name"]),
        left_on="To Account",
        right_on="Account Number",
        how="left",
        suffix="_receiver"
    )
    .rename({
        "Entity Name": "Receiver_Entity",
        "Bank Name": "Receiver_Bank_Name"
    })

    # (선택) rate 컬럼 정리하고 싶으면:
   .drop(["recv_rate","pay_rate"])
)

In [ ]:
def parse_aml_patterns(file_path):
    data = []
    # 패턴명과 Max Hops/Degree 정보를 추출하기 위한 정규표현식
    # 예: "BEGIN LAUNDERING ATTEMPT - CYCLE:  Max 12 hops"
    pattern_re = re.compile(r"BEGIN LAUNDERING ATTEMPT\s*-\s*([^:]+)(?::\s*(.*))?")

    current_pattern = "NONE"
    current_meta = ""

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            # 1. 패턴 시작점 파싱
            if "BEGIN LAUNDERING ATTEMPT" in line:
                match = pattern_re.search(line)
                if match:
                    current_pattern = match.group(1).strip() # 예: CYCLE
                    current_meta = match.group(2).strip() if match.group(2) else "" # 예: Max 12 hops
                continue

            # 2. 패턴 종료점
            if "END LAUNDERING ATTEMPT" in line:
                current_pattern = "NONE"
                current_meta = ""
                continue

            # 3. 거래 데이터 라인 (CSV 형태)
            # 형식: Timestamp, From_ID, From_Acc, To_ID, To_Acc, Amount, Currency, ...
            if current_pattern != "NONE" and "," in line:
                parts = line.split(",")
                if len(parts) >= 9:
                    data.append({
                        "Timestamp": parts[0].strip(),
                        "From Account": parts[2].strip(), # 814167590 같은 계좌번호
                        "To Account": parts[4].strip(),
                        "Amount Received": float(parts[5].strip()),
                        "Pattern_Type": current_pattern,
                        "Pattern_Detail": current_meta # Max hops 정보 저장
                    })

    return pl.DataFrame(data)

# --- 실행부 ---
df_patterns = parse_aml_patterns(patterns_txt)

# q_master와 조인 (Timestamp, 계좌번호, 금액 3중 키로 정확도 확보)
# 주의: q_master의 Timestamp와 Amount 타입을 패턴 데이터와 맞춰야 합니다.
q_master = q_master.join(
    df_patterns.lazy(),
    on=["Timestamp", "From Account", "To Account", "Amount Received"],
    how="left"
).with_columns([
    pl.col("Pattern_Type").fill_null("NONE"),
    pl.col("Pattern_Detail").fill_null("")
])

In [ ]:
# -----------------------------
#  sink to Parquet (Streaming write)
# -----------------------------
print("Writing master table to Parquet (streaming)...")
q_master.sink_parquet(master_parquet)
print(f"Done. Saved: {master_parquet}")

# 이후에는 이렇게 빠르게 로드
# q_master = pl.scan_parquet(master_parquet)

Writing master table to Parquet (streaming)...
Done. Saved: /content/drive/MyDrive/data/parquet/HI-Medium_Master.parquet


In [ ]:
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master.parquet"

q_master = pl.scan_parquet(master_parquet)

# 3) 은행 -> 국가 관련 전처리


In [ ]:
# -----------------------------------------------------------------------------
# 1. 유니크 은행 명단 추출 (Lazy -> Collect)
# -----------------------------------------------------------------------------
# base_path가 정의되어 있다고 가정합니다.
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master.parquet"

print("1. 유니크 은행 명단 추출 중...")
q_master = pl.scan_parquet(master_parquet)

# Sender_Bank_Name만 유니크하게 뽑아옵니다.
banks = (
    q_master
    .select(pl.col("Sender_Bank_Name"))
    .unique()
    .collect()
)

print(f"   - 총 유니크 은행 수: {len(banks)}")

1. 유니크 은행 명단 추출 중...
   - 총 유니크 은행 수: 80179


In [ ]:
# -----------------------------------------------------------------------------
# 2. 국가/도시 매핑 사전(Dictionary) 구축 (Lookup Table)
# -----------------------------------------------------------------------------
print("2. 매핑 사전(Lookup Table) 구축 중...")

gc = geonamescache.GeonamesCache()
countries = gc.get_countries()
cities = gc.get_cities()

# 검색용 딕셔너리: { "검색어(소문자)": "표준 국가명" }
location_map = {}

# 국가 이름 및 ISO 코드 등록
for code, data in countries.items():
    country_name = data['name']
    location_map[country_name.lower()] = country_name
    location_map[code.lower()] = country_name # ISO 2자리 코드 (US, KR 등)

# (pycountry 보완 (공식 명칭 등)
for country in pycountry.countries:
    location_map[country.name.lower()] = country.name
    try:
        # official_name이 있는 경우
        if hasattr(country, 'official_name'):
            location_map[country.official_name.lower()] = country.name
    except:
        pass

# (3) 도시 이름 -> 국가 이름 매핑
# 동명이인 도시(예: Paris, Texas vs Paris, France) 처리: 인구가 많은 쪽 우선
city_pop_map = {} # { "city_name": max_population }

for _, data in cities.items():
    city_name = data['name'].lower()
    country_code = data['countrycode']
    population = data.get('population', 0)

    # 해당 국가 코드가 우리 국가 사전에 있어야 함
    if country_code in countries:
        country_name = countries[country_code]['name']

        # 이미 등록된 도시라면 인구수 비교해서 갱신
        if city_name not in city_pop_map or population > city_pop_map[city_name]:
            city_pop_map[city_name] = population
            location_map[city_name] = country_name

# (4) [중요] 사용자 정의 별칭(Alias) 및 예외 처리 등록
custom_aliases = {
    "usa": "United States",
    "us": "United States",
    "america": "United States",
    "uk": "United Kingdom",
    "britain": "United Kingdom",
    "england": "United Kingdom",
    "uae": "United Arab Emirates",
    "korea": "South Korea",
    "south korea": "South Korea",
    "russia": "Russia",
    "vietnam": "Vietnam",
    "deutschland": "Germany",
    # (주의) Saudi Arabia와 Crypto는 함수 로직에서 우선 처리하지만 맵에도 추가
    "saudi": "Saudi Arabia",
    "arabia": "Saudi Arabia",
}
location_map.update(custom_aliases)

2. 매핑 사전(Lookup Table) 구축 중...


In [ ]:
# -----------------------------------------------------------------------------
# 3. 국가 추출 함수 (로직 통합 및 강화)
# -----------------------------------------------------------------------------
def extract_country_robust(bank_name):
    if bank_name is None:
        return "United States" # 기본값

    # 1. 전처리: 소문자 변환 및 특수문자 제거 후 공백 정리
    # "Saudi Arabia Bank #123" -> "saudi arabia bank 123"
    name_clean = re.sub(r"[^a-zA-Z0-9\s]", " ", str(bank_name).lower())
    name_clean = re.sub(r"\s+", " ", name_clean).strip()

    # 2. [최우선 규칙] 특정 키워드가 포함되면 즉시 반환 (contains 로직)
    # Saudi Arabia 처리
    if "saudi arabia" in name_clean:
        return "Saudi Arabia"

    # Crypto 및 오타(Crytpo) 처리
    if "crypto" in name_clean or "crytpo" in name_clean:
        return "Crypto"  # 국가명 대신 'Crypto'라는 카테고리로 분류

    # 3. 토큰 단위 매칭 (단어 쪼개기)
    tokens = name_clean.split()

    # (A) "Bank of {Location}" 패턴 확인 (정확도 높음)
    # 예: Bank of Dallas -> Dallas 검색
    if len(tokens) >= 3 and tokens[0] == "bank" and tokens[1] == "of":
        candidate = tokens[2]
        if candidate in location_map:
            return location_map[candidate]

    # (B) 일반 토큰 검색
    # 문자열의 뒤쪽(도시/국가가 보통 뒤에 옴)부터 검색할 수도 있지만, 여기선 순차 검색
    for token in tokens:
        # "Bank", "International" 같은 일반 명사는 제외해야 함 (사전에 없다고 가정하거나 필터링)
        if token in ["bank", "central", "national", "union", "credit", "unknown"]:
            continue

        if token in location_map:
            return location_map[token]

    # 4. [Fallback] 아무것도 못 찾았을 경우
    # 질문 내용상 "나머지 null은 미국 지역은행"이라고 가정하므로 United States 반환
    return "United States"

In [ ]:
# -----------------------------------------------------------------------------
# 4. 데이터프레임에 적용 (map_elements 활용)
# -----------------------------------------------------------------------------
print("3. 국가 추출 로직 적용 중...")

# map_elements로 함수 적용 (return_dtype 지정 필수)
bank_mapping_df = banks.with_columns(
    pl.col("Sender_Bank_Name")
    .map_elements(extract_country_robust, return_dtype=pl.String)
    .alias("From Country")
)

3. 국가 추출 로직 적용 중...


In [ ]:
# -----------------------------------------------------------------------------
# 5. 검증 (Validation)
# -----------------------------------------------------------------------------
print("\n=== 검증 리포트 ===")

# 1) Null 확인 (로직상 0이어야 함)
null_count = bank_mapping_df.filter(pl.col("From Country").is_null()).height
print(f"1. 최종 Null 개수: {null_count} (목표: 0)")

# 2) Saudi Arabia 확인
saudi_count = bank_mapping_df.filter(pl.col("From Country") == "Saudi Arabia").height
print(f"2. Saudi Arabia 추출 개수: {saudi_count}")

# 3) Crypto (오타 포함) 확인
crypto_count = bank_mapping_df.filter(pl.col("From Country") == "Crypto").height
print(f"3. Crypto 추출 개수: {crypto_count}")

# 4) United States 비율 확인 (Fallback이 잘 작동했는지)
us_count = bank_mapping_df.filter(pl.col("From Country") == "United States").height
print(f"4. United States (Fallback 포함) 개수: {us_count}")


=== 검증 리포트 ===
1. 최종 Null 개수: 0 (목표: 0)
2. Saudi Arabia 추출 개수: 2397
3. Crypto 추출 개수: 2723
4. United States (Fallback 포함) 개수: 262


In [ ]:
# -----------------------------------------------------------------------------
# 6. 원본 데이터와 조인 및 저장
# -----------------------------------------------------------------------------
print("\n4. 원본 데이터와 조인 및 저장 중...")

# LazyFrame으로 변환하여 조인 준비
mapping_lf = bank_mapping_df.lazy()

final_lf = (
    q_master
    .join(
        mapping_lf,
        on="Sender_Bank_Name",
        how="left"
    )
)

# # 파일 저장 (Parquet)
# output_path = "q_master_with_country.parquet"
# final_lf.sink_parquet(output_path)

# print(f"완료! 파일 저장됨: {output_path}")


4. 원본 데이터와 조인 및 저장 중...


In [ ]:
# # 1. 파일 불러오기 (Lazy)
# check_lf = pl.scan_parquet("q_master_with_country.parquet")

# 2. 유니크 은행(Sender_Bank_Name) 기준으로 중복 제거 후 집계
# 각 국가별로 '몇 종류의 은행'이 있는지 계산합니다.
bank_dist_report = (
    final_lf
    .select(["Sender_Bank_Name", "From Country"])
    .unique() # <--- 여기서 은행 이름 중복을 제거합니다.
    .group_by("From Country")
    .agg(pl.len().alias("unique_bank_count"))
    .sort("unique_bank_count", descending=True)
    .collect(engine="streaming")
)

print("=== [은행 종류 기준] 국가별 매핑 분포 ===")
print(bank_dist_report.to_pandas().to_string(index=False))

# 3. 만약 여전히 null이 있다면, 어떤 은행들이 null인지 상위 20개 확인
null_banks_sample = (
    final_lf
    .select(["Sender_Bank_Name", "From Country"])
    .unique()
    .filter(pl.col("From Country").is_null())
    .collect(engine="streaming")
)

print(f"\n매핑되지 않은(null) 유니크 은행 수: {len(null_banks_sample)}개")
if len(null_banks_sample) > 0:
    print("--- 매핑 실패 은행 샘플 ---")
    print(null_banks_sample.head(20))

=== [은행 종류 기준] 국가별 매핑 분포 ===
  From Country  unique_bank_count
         China               8967
       Germany               6222
        Israel               6172
        France               5042
         Italy               4508
        Canada               4398
United Kingdom               4189
        Russia               4181
     Australia               3910
   Switzerland               3657
         Spain               3645
   Philippines               3608
         Japan               3452
         India               3139
        Crypto               2723
        Brazil               2680
  Saudi Arabia               2397
   Netherlands               1299
       Belgium                878
      Portugal                837
        Greece                810
       Austria                678
      Slovakia                422
       Finland                420
       Ireland                376
       Croatia                317
 United States                262
     Lithuania     

In [ ]:
def create_bank_mapping_df(unique_banks_list):
    """
    유니크 은행 리스트를 받아 국가 정보를 매핑한 DF 반환 (보정 규칙 포함)
    """
    # --- [A] 기초 데이터 로드 ---
    gc = geonamescache.GeonamesCache()
    cities = gc.get_cities()
    city_to_country = {info["name"].upper(): info["countrycode"] for _, info in cities.items()}
    cc_to_name = {c.alpha_2: c.name for c in pycountry.countries}
    country_one_tokens = {c.name.upper() for c in pycountry.countries if len(re.findall(r"[A-Z]+", c.name.upper())) == 1}

    # [수정] ALIASES에 TÜRKIYE와 TURKIYE 추가
    ALIASES = {
        "US": "United States", "USA": "United States", "UAE": "United Arab Emirates",
        "UK": "United Kingdom", "KOREA": "Korea, Republic of", "RUSSIA": "Russian Federation",
        "TÜRKIYE": "Turkey", "TURKIYE": "Turkey"
    }

    def extract_country_internal(bank_name):
        s = "" if bank_name is None else str(bank_name).upper()
        toks = re.sub(r"[^A-Z]+", " ", s).strip().split()
        if not toks: return None
        # {X} BANK 패턴
        if len(toks) >= 2 and toks[1] == "BANK":
            x = toks[0]
            if x in ALIASES: return ALIASES[x]
            if x in country_one_tokens: return x.title()
        # BANK OF {CITY} 패턴
        if len(toks) >= 3 and toks[0] == "BANK" and toks[1] == "OF":
            cc = city_to_country.get(toks[2])
            if cc: return cc_to_name.get(cc, cc)
        # 전체 토큰 매칭
        for t in toks:
            if t in ALIASES: return ALIASES[t]
            if t in country_one_tokens: return t.title()
            cc = city_to_country.get(t)
            if cc: return cc_to_name.get(cc, cc)
        return None

    # --- [B] 기초 매핑 생성 ---
    # 파이썬 리스트를 먼저 생성한 후, 명시적으로 Series로 변환하여 길이를 맞춥니다.
    # 만약 데이터프레임이 들어오면 첫 번째 컬럼을 리스트로 변환
    if isinstance(unique_banks_list, pl.DataFrame):
        input_list = unique_banks_list.to_series(0).to_list()
    elif isinstance(unique_banks_list, pl.Series):
        input_list = unique_banks_list.to_list()
    else:
        input_list = list(unique_banks_list)

    # 이제 80179개를 제대로 돕니다.
    country_raw_list = [extract_country_internal(b) for b in input_list]

    temp_df = pl.DataFrame({
        "Bank_Name_Key": input_list,
        "Country_Raw": pl.Series(country_raw_list, dtype=pl.Utf8)
    })

    # --- [C] 규칙 보정 (Saudi Arabia, Crypto, Typos, Turkey) ---
    name_norm = (
        pl.col("Bank_Name_Key").cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^a-z0-9]+", " ")
        .str.strip_chars()
        .str.replace_all(r"\s+", " ")
    )

    # [수정] rules에 türkiye와 turkiye 추가
    rules = [
        ("saudi arabia", "Saudi Arabia"),
        ("crypto", "Crypto"),
        ("crytpo", "Crypto"),
        ("türkiye", "Turkey"),
        ("turkiye", "Turkey")
    ]

    # 시작점을 Country_Raw로 잡아서 길이를 맞춤
    final_expr = pl.col("Country_Raw")

    # 규칙 표현식 생성
    for k, v in rules:
        # 기존 값이 없거나(null), 특정 키워드가 포함된 경우 보정
        final_expr = (
            pl.when(name_norm.str.contains(k, literal=True))
            .then(pl.lit(v))
            .otherwise(final_expr)
        )

    # --- [D] 최종 매핑 테이블 완성 ---
    bank_mapping_df = temp_df.select([
        pl.col("Bank_Name_Key").alias("Mapping_Bank_Name"),
        final_expr
          .str.replace("Türkiye", "Turkey") # 예외 처리
          .fill_null("United States")      # 마지막에 US 채우기
          .alias("Mapped_Country")
    ])

    return bank_mapping_df # 할당 연산자(:=) 제거하고 깔끔하게 리턴

bank_mapping_df = create_bank_mapping_df(banks)  # ✅ 이 줄이 꼭 있어야 함
bank_mapping_df.head(5)

Mapping_Bank_Name,Mapped_Country
str,str
"""Switzerland Bank #337""","""Switzerland"""
"""Israel Bank #3229""","""Israel"""
"""Crytpo Bank #271""","""Crypto"""
"""UK Bank #1970""","""United Kingdom"""
"""Israel Bank #1658""","""Israel"""


In [ ]:
def enrich_master_with_countries(q_master, bank_mapping_df):
    FATF_MEMBERS = [
        "Argentina", "Australia", "Austria", "Belgium", "Brazil", "Canada", "China",
        "Denmark", "Finland", "France", "Germany", "Greece", "Hong Kong", "Iceland",
        "India", "Ireland", "Israel", "Italy", "Japan", "Korea, Republic of",
        "Luxembourg", "Malaysia", "Mexico", "Netherlands", "New Zealand", "Norway",
        "Portugal", "Saudi Arabia", "Singapore", "South Africa", "Spain", "Sweden",
        "Switzerland", "Turkey", "United Kingdom", "United States"
    ]

    # 1. 매핑 테이블 준비
    s_map = bank_mapping_df.select([
        pl.col("Mapping_Bank_Name").alias("Sender_Bank_Name"),
        pl.col("Mapped_Country").alias("From Country")
    ])

    r_map = bank_mapping_df.select([
        pl.col("Mapping_Bank_Name").alias("Receiver_Bank_Name"),
        pl.col("Mapped_Country").alias("To Country")
    ])

    # 2. 조인 수행
    q_enriched = (
        q_master
        .join(s_map.lazy(), on="Sender_Bank_Name", how="left")
        .join(r_map.lazy(), on="Receiver_Bank_Name", how="left")
    )

    # 3. 피쳐 생성 및 보정
    q_enriched = q_enriched.with_columns([
        # [수정] 여기서도 혹시 모를 "Türkiye"를 한 번 더 방어적으로 Turkey로 바꿈
        pl.col("From Country").fill_null("United States").str.replace("Türkiye", "Turkey"),
        pl.col("To Country").fill_null("United States").str.replace("Türkiye", "Turkey")
    ]).with_columns([
        (pl.col("From Country") != pl.col("To Country")).cast(pl.Boolean).alias("IsCrossBorder"),
        pl.col("From Country").is_in(FATF_MEMBERS).cast(pl.Boolean).alias("IsFromFATF"),
        pl.col("To Country").is_in(FATF_MEMBERS).cast(pl.Boolean).alias("IsToFATF")
    ])

    return q_enriched # 연산 계획이 담긴 LazyFrame 반환

q_master_enriched = enrich_master_with_countries(q_master, bank_mapping_df)  # ✅

In [ ]:
# 4. 검증 함수 실행 (이제 'From Country'가 포함된 q_master를 전달합니다)
def validate_final_mapping(lf):
    # 컬럼이 있는지 먼저 확인하는 안전 장치 추가
    cols = lf.collect_schema().names()
    if "From Country" not in cols:
        print("❌ 에러: From Country 컬럼이 생성되지 않았습니다. 조인 과정을 확인하세요.")
        return

    stats = lf.select([
        pl.len().alias("Total"),
        pl.col("From Country").is_null().sum().alias("Nulls"),
        (pl.col("From Country") == "Saudi Arabia").sum().alias("Saudi_Count"),
        (pl.col("From Country") == "Crypto").sum().alias("Crypto_Count"),
        pl.col("IsCrossBorder").sum().alias("CrossBorder_Sum"),
        pl.col("IsCrossBorder").sum().alias("CrossBorder_Count")
    ]).collect()

    print(f"Total Transactions: {stats['Total'][0]:,}")
    print(f"Remaining Nulls: {stats['Nulls'][0]}")
    print(f"CrossBorder: {stats['CrossBorder_Count'][0]:,}")
    print(f"Saudi Arabia Transactions: {stats['Saudi_Count'][0]:,}")
    print(f"Crypto Transactions: {stats['Crypto_Count'][0]:,}")
    print(f"Cross-Border Transactions: {stats['CrossBorder_Sum'][0]:,}")

validate_final_mapping(q_master_enriched)  # ✅ q_master 말고!

Total Transactions: 31,898,669
Remaining Nulls: 0
CrossBorder: 13,910,209
Saudi Arabia Transactions: 420,142
Crypto Transactions: 552,010
Cross-Border Transactions: 13,910,209


In [ ]:
def _normalize_bank_mapping_df(bank_mapping_df: pl.DataFrame) -> pl.DataFrame:
    """
    bank_mapping_df가 어떤 버전이든(스키마가 달라도) 아래 표준 스키마로 통일해서 반환:
      - Sender_Bank_Name
      - From Country
    지원 스키마:
      A) ["Sender_Bank_Name", "From Country"]
      B) ["Mapping_Bank_Name", "Mapped_Country"]
    """
    cols = set(bank_mapping_df.columns)

    # 버전 A
    if {"Sender_Bank_Name", "From Country"}.issubset(cols):
        out = bank_mapping_df.select([
            pl.col("Sender_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
            pl.col("From Country").cast(pl.Utf8).str.strip_chars().alias("From Country"),
        ])
        return out

    # 버전 B
    if {"Mapping_Bank_Name", "Mapped_Country"}.issubset(cols):
        out = bank_mapping_df.select([
            pl.col("Mapping_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
            pl.col("Mapped_Country").cast(pl.Utf8).str.strip_chars().alias("From Country"),
        ])
        return out

    raise ValueError(
        f"[bank_mapping_df 스키마 인식 실패] columns={bank_mapping_df.columns}\n"
        "필요 컬럼: (Sender_Bank_Name, From Country) 또는 (Mapping_Bank_Name, Mapped_Country)"
    )


def add_aml_features(q_master: pl.LazyFrame, bank_mapping_df: pl.DataFrame) -> pl.LazyFrame:
    """
    1) Sender/Receiver 은행명 -> From/To Country 매핑 (bank_mapping_df 버전 A/B 자동 대응)
    2) Null/Unknown -> United States 보정
    3) IsCrossBorder, IsFromFATF, IsToFATF 피쳐 생성
    """

    FATF_MEMBERS = [
        "Argentina", "Australia", "Austria", "Belgium", "Brazil", "Canada", "China",
        "Denmark", "Finland", "France", "Germany", "Greece", "Hong Kong", "Iceland",
        "India", "Ireland", "Israel", "Italy", "Japan", "Korea, Republic of",
        "Luxembourg", "Malaysia", "Mexico", "Netherlands", "New Zealand", "Norway",
        "Portugal", "Saudi Arabia", "Singapore", "South Africa", "Spain", "Sweden",
        "Switzerland", "Turkey", "United Kingdom", "United States"
    ]

    # ✅ bank_mapping_df 어떤 버전이 와도 표준 스키마로 통일
    mapping_std = _normalize_bank_mapping_df(bank_mapping_df)

    # sender mapping: Sender_Bank_Name -> From Country
    sender_map = mapping_std.select([
        pl.col("Sender_Bank_Name"),
        pl.col("From Country"),
    ])

    # receiver mapping: Receiver_Bank_Name -> To Country
    receiver_map = mapping_std.select([
        pl.col("Sender_Bank_Name").alias("Receiver_Bank_Name"),
        pl.col("From Country").alias("To Country"),
    ])

    # ✅ join key도 공백/NULL 방어 (조인 실패 줄이기)
    q_master = q_master.with_columns([
        pl.col("Sender_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
        pl.col("Receiver_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Receiver_Bank_Name"),
    ])

    # 1) 국가 조인
    q_master = (
        q_master
        .join(sender_map.lazy(), on="Sender_Bank_Name", how="left")
        .join(receiver_map.lazy(), on="Receiver_Bank_Name", how="left")
    )

    # 2) 보정: null / "Unknown" -> United States
    def _fix_unknown(expr: pl.Expr) -> pl.Expr:
        return (
            expr
            .cast(pl.Utf8)
            .str.strip_chars()
            .fill_null("United States")
            .pipe(lambda e: pl.when(e.str.to_lowercase() == "unknown").then(pl.lit("United States")).otherwise(e))
        )

    q_master = q_master.with_columns([
        _fix_unknown(pl.col("From Country")).alias("From Country"),
        _fix_unknown(pl.col("To Country")).alias("To Country"),
    ])

    # 3) AML 피쳐 생성
    q_master = q_master.with_columns([
        (pl.col("From Country") != pl.col("To Country")).cast(pl.UInt8).alias("IsCrossBorder"),
        pl.col("From Country").is_in(FATF_MEMBERS).cast(pl.UInt8).alias("IsFromFATF"),
        pl.col("To Country").is_in(FATF_MEMBERS).cast(pl.UInt8).alias("IsToFATF"),
    ])

    return q_master

q_master_enriched = add_aml_features(q_master, bank_mapping_df)

In [ ]:
def validate_aml_features(q_master_final):
    print("=== [검증 시작] AML 피쳐 생성 결과 확인 ===")

    # 1. 기초 통계량 (전체 건수 및 피쳐별 합계)
    stats = q_master_final.select([
        pl.len().alias("Total_Rows"),
        pl.col("IsCrossBorder").sum().alias("CrossBorder_Count"),
        pl.col("IsFromFATF").sum().alias("FromFATF_Count"),
        pl.col("IsToFATF").sum().alias("ToFATF_Count")
    ]).collect()

    total = stats['Total_Rows'][0]
    print(f"전체 거래 건수: {total:,}")
    print(f"해외 송금 건수(CrossBorder): {stats['CrossBorder_Count'][0]:,} ({(stats['CrossBorder_Count'][0]/total*100):.2f}%)")
    print(f"송신국 FATF 가입 건수: {stats['FromFATF_Count'][0]:,} ({(stats['FromFATF_Count'][0]/total*100):.2f}%)")
    print(f"수신국 FATF 가입 건수: {stats['ToFATF_Count'][0]:,} ({(stats['ToFATF_Count'][0]/total*100):.2f}%)")

    # 2. 크로스 체크: 해외 송금인데 사기 거래(Is Laundering == 1)인 비중
    fraud_cross = q_master_final.filter((pl.col("IsCrossBorder") == 1) & (pl.col("Is Laundering") == 1)).select(pl.len()).collect().item()
    print(f"해외 송금 중 사기 거래 건수: {fraud_cross:,}")

    # 3. 샘플 확인
    print("\n=== 데이터 샘플 (AML 피쳐 중심) ===")
    print(q_master_final.select([
        "From Country", "To Country", "IsCrossBorder", "IsFromFATF", "IsToFATF"
    ]).head(5).collect())

validate_aml_features(q_master_enriched)

=== [검증 시작] AML 피쳐 생성 결과 확인 ===
전체 거래 건수: 31,898,669
해외 송금 건수(CrossBorder): 13,910,209 (43.61%)
송신국 FATF 가입 건수: 30,132,384 (94.46%)
수신국 FATF 가입 건수: 29,886,763 (93.69%)
해외 송금 중 사기 거래 건수: 24,750

=== 데이터 샘플 (AML 피쳐 중심) ===
shape: (5, 5)
┌──────────────┬────────────┬───────────────┬────────────┬──────────┐
│ From Country ┆ To Country ┆ IsCrossBorder ┆ IsFromFATF ┆ IsToFATF │
│ ---          ┆ ---        ┆ ---           ┆ ---        ┆ ---      │
│ str          ┆ str        ┆ u8            ┆ u8         ┆ u8       │
╞══════════════╪════════════╪═══════════════╪════════════╪══════════╡
│ France       ┆ France     ┆ 0             ┆ 1          ┆ 1        │
│ Canada       ┆ Canada     ┆ 0             ┆ 1          ┆ 1        │
│ Canada       ┆ Canada     ┆ 0             ┆ 1          ┆ 1        │
│ Germany      ┆ Germany    ┆ 0             ┆ 1          ┆ 1        │
│ Germany      ┆ Germany    ┆ 0             ┆ 1          ┆ 1        │
└──────────────┴────────────┴───────────────┴────────────┴───────

In [ ]:
def extract_branch_ids_offset(q_master):
    return q_master.with_columns([
        # 1. 숫자를 추출합니다 (추출 실패 시 null)
        pl.col("Sender_Bank_Name")
        .str.extract(r"#\s*(\d+)", 1)
        .cast(pl.UInt32)
        .alias("s_tmp"),

        pl.col("Receiver_Bank_Name")
        .str.extract(r"#\s*(\d+)", 1)
        .cast(pl.UInt32)
        .alias("r_tmp")
    ]).with_columns([
        # 2. 제안하신 로직 적용:
        # 숫자가 있으면(not null) 값 + 1, 없으면(null) 0
        pl.when(pl.col("s_tmp").is_not_null())
          .then(pl.col("s_tmp") + 1)
          .otherwise(0)
          .cast(pl.UInt32)
          .alias("Sender_Bank_Branch_ID"),

        pl.when(pl.col("r_tmp").is_not_null())
          .then(pl.col("r_tmp") + 1)
          .otherwise(0)
          .cast(pl.UInt32)
          .alias("Receiver_Bank_Branch_ID")
    ]).drop(["s_tmp", "r_tmp"]) # 임시 컬럼 삭제

# 실행
q_master_enriched = extract_branch_ids_offset(q_master_enriched)

In [ ]:
# 검증을 위한 샘플 출력
validation_check = (
    q_master_enriched
    .select(["Sender_Bank_Name", "Sender_Bank_Branch_ID"])
    .unique()
    .filter(
        # 세 가지 케이스를 골고루 보기 위한 필터
        (pl.col("Sender_Bank_Name").str.contains("#0")) |    # 케이스 1: #0 -> 1이 되었는가?
        (~pl.col("Sender_Bank_Name").str.contains("#")) |    # 케이스 2: #없음 -> 0이 되었는가?
        (pl.col("Sender_Bank_Name").str.contains("#100"))    # 케이스 3: #100 -> 101이 되었는가?
    )
    .head(10)
    .collect(engine="streaming")
)

print("=== 지점 ID 오프셋 적용 결과 ===")
print(validation_check)

# 통계 확인
branch_stats = (
    q_master_enriched
    .group_by("Sender_Bank_Branch_ID")
    .agg(pl.len().alias("count"))
    .sort("Sender_Bank_Branch_ID")
    .head(5)
    .collect(engine="streaming")
)
print("\n=== ID별 거래 분포 (0:단일, 1:본점, 2+:지점) ===")
print(branch_stats)

=== 지점 ID 오프셋 적용 결과 ===
shape: (10, 2)
┌────────────────────────────┬───────────────────────┐
│ Sender_Bank_Name           ┆ Sender_Bank_Branch_ID │
│ ---                        ┆ ---                   │
│ str                        ┆ u32                   │
╞════════════════════════════╪═══════════════════════╡
│ National Bank of Chicago   ┆ 0                     │
│ Hearthstone Community Bank ┆ 0                     │
│ Bank of Helena             ┆ 0                     │
│ Sappo Savings Bank         ┆ 0                     │
│ Italy Bank #10017          ┆ 10018                 │
│ Hilltop Bancorp            ┆ 0                     │
│ Dawn Federal Bank          ┆ 0                     │
│ Regents Federal Bank       ┆ 0                     │
│ First Bank of Tampa        ┆ 0                     │
│ Willows Trust Bank         ┆ 0                     │
└────────────────────────────┴───────────────────────┘

=== ID별 거래 분포 (0:단일, 1:본점, 2+:지점) ===
shape: (5, 2)
┌───────────────────────┬───

In [ ]:
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet"
print("Writing:", master_parquet)
q_master_enriched.sink_parquet(master_parquet)
print("Done!")

Writing: /content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet
Done!


In [ ]:
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet"

q_master = pl.scan_parquet(master_parquet)

# 4) Timestamp

In [ ]:
# =========================================================
# Timestamp 파싱 + 정렬 준비
#   - 원본 Timestamp(string) 유지
#   - ts(Datetime) 새로 생성
#   - ts 파싱 실패(null) 개수 출력
# =========================================================
total_rows = q_master.select(pl.len()).collect().item()

q_master = q_master.with_columns(
    pl.col("Timestamp")
      .str.strptime(pl.Datetime, format="%Y/%m/%d %H:%M", strict=False)
      .alias("ts")
)

null_ts_rows = (
    q_master.filter(pl.col("ts").is_null())
     .select(pl.len())
     .collect()
     .item()
)

valid_rows = total_rows - null_ts_rows

print(f"Total rows                    : {total_rows:,}")
print(f"Rows with NULL ts (parse fail): {null_ts_rows:,}")
print(f"Valid ts rows                 : {valid_rows:,}")

# split은 ts가 유효한 행만 대상으로 진행 (추천)
q_valid = q_master.filter(pl.col("ts").is_not_null())

Total rows                    : 31,898,669
Rows with NULL ts (parse fail): 0
Valid ts rows                 : 31,898,669


In [ ]:
# =========================================================
# time-based split 기준(ts cut) 계산 (60:20:20)
#   - collect 최소화: 경계 ts 2개만 뽑음
# =========================================================
n = q_valid.select(pl.len()).collect().item()
train_end_idx = int(n * 0.6)
val_end_idx   = int(n * 0.8)

q_sorted = q_valid.sort("ts")

train_cut_ts = (
    q_sorted.select("ts")
            .slice(train_end_idx, 1)
            .collect()
            .item()
)

val_cut_ts = (
    q_sorted.select("ts")
            .slice(val_end_idx, 1)
            .collect()
            .item()
)

print("Train cutoff ts:", train_cut_ts)
print("Val cutoff ts  :", val_cut_ts)

Train cutoff ts: 2022-09-09 20:43:00
Val cutoff ts  : 2022-09-14 05:46:00


In [ ]:
# =========================================================
# split 컬럼 부여 (train/val/test)
# =========================================================
q_split = q_sorted.with_columns(
    pl.when(pl.col("ts") <= train_cut_ts)
      .then(pl.lit("train"))
      .when(pl.col("ts") <= val_cut_ts)
      .then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("split")
)

# sanity check: split 분포
split_counts = (
    q_split.group_by("split")
           .agg(pl.len().alias("rows"))
           .collect()
)
print(split_counts)

# 라벨 비율 sanity check (optional)
label_counts = (
    q_split.group_by(["split", "Is Laundering"])
           .agg(pl.len().alias("rows"))
           .collect()
)
print(label_counts)

shape: (3, 2)
┌───────┬──────────┐
│ split ┆ rows     │
│ ---   ┆ ---      │
│ str   ┆ u32      │
╞═══════╪══════════╡
│ test  ┆ 6378958  │
│ val   ┆ 6379980  │
│ train ┆ 19139731 │
└───────┴──────────┘
shape: (6, 3)
┌───────┬───────────────┬──────────┐
│ split ┆ Is Laundering ┆ rows     │
│ ---   ┆ ---           ┆ ---      │
│ str   ┆ i64           ┆ u32      │
╞═══════╪═══════════════╪══════════╡
│ test  ┆ 0             ┆ 6368318  │
│ test  ┆ 1             ┆ 10640    │
│ train ┆ 1             ┆ 15536    │
│ train ┆ 0             ┆ 19124195 │
│ val   ┆ 0             ┆ 6370926  │
│ val   ┆ 1             ┆ 9054     │
└───────┴───────────────┴──────────┘


# 6) bucket_ts 생성 (1h)

In [ ]:
# =========================================================
# bucket_ts 생성 (ts → 1h truncate)
#   - baseline v1: 거래(row) 단위 모델링이지만,
#     이후 확장을 위해 bucket_ts는 만들어 둠
# =========================================================
q_feat = q_split.with_columns(
    pl.col("ts").dt.truncate("1h").alias("bucket_ts")
).with_columns(
    pl.col("bucket_ts").dt.strftime("%Y-%m-%d %H:%M:%S").alias("bucket_ts_str")
)

# bucket 분포 sanity check (optional)
bucket_check = (
    q_feat.filter(pl.col("split") == "train")
          .select([
              pl.col("bucket_ts").min().alias("train_bucket_min"),
              pl.col("bucket_ts").max().alias("train_bucket_max"),
              pl.len().alias("train_rows"),
          ])
          .collect()
)
print(bucket_check)

shape: (1, 3)
┌─────────────────────┬─────────────────────┬────────────┐
│ train_bucket_min    ┆ train_bucket_max    ┆ train_rows │
│ ---                 ┆ ---                 ┆ ---        │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32        │
╞═════════════════════╪═════════════════════╪════════════╡
│ 2022-09-01 00:00:00 ┆ 2022-09-09 20:00:00 ┆ 19139731   │
└─────────────────────┴─────────────────────┴────────────┘


In [ ]:
#“노드 키” 만들기 (송신/수신 각각)
q_feat = q_feat.with_columns([
    (pl.col("From Account").cast(pl.Utf8) + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("sender_node"),
    (pl.col("To Account").cast(pl.Utf8)   + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("receiver_node"),
])

# 8-1) 기존 피처 엔지니어링

In [ ]:
# -------------------------
# (A) 상수/룩업 리스트
# -------------------------
PAYMENT_FORMATS = ["ACH", "Cheque", "Bitcoin", "Cash", "Credit Card", "Wire", "Reinvestment"]

HIGH_RISK_SENDER_BANKS = [
    "National Bank of Dallas", "Savings Bank of Augusta", "China Bank #27",
    "India Bank #96", "Brazil Bank #128", "National Bank of Milford",
    "Savings Bank of Sacramento", "Saudi Arabia Bank #14", "Israel Bank #16",
    "Golden Credit Union"
]

HIGH_RISK_RECEIVER_BANKS = [
    "China Bank #292", "China Bank #27", "China Bank #22", "Japan Bank #143",
    "Brazil Bank #50", "Bank of Denver", "Saudi Arabia Bank #56",
    "Israel Bank #48", "The Pine Bank", "National Bank of Milford"
]

# 엔티티 타입 가중치 (v1: 단순 룰)
ENTITY_TYPE_SCORE_MAP = {
    "Corporation": 2.0,
    "Sole Proprietorship": 1.5
}

# Payment method risk (v1: 단순 룰)
# (정확한 가중치는 EDA에서 조정 가능. 지금은 “비중 높았던 수단에 가중”만 구현)
PAYMENT_RISK_MAP = {
    "ACH": 3.0,
    "Bitcoin": 2.5,
    "Cheque": 2.5,
    "Cash": 1.5,
    "Credit Card": 1.3,
    "Wire": 1.0,
    "Reinvestment": 1.0
}

In [ ]:
# -------------------------
# (B) Timestamp 파생 + (추가) hour_sin/cos + dow_cat
#  - hour/day_of_week는 유지(다른 코드 깨짐 방지)
#  - hour_sin/cos: 주기성 반영
#  - dow_cat: 요일을 "순서 없는 범주"로 명시 (필요시 XGBoost categorical용)
# -------------------------
q_feat = q_feat.with_columns([
    # 기존 파생 (그대로 유지)
    pl.col("ts").dt.hour().cast(pl.Int16).alias("hour"),
    pl.col("ts").dt.weekday().cast(pl.Int8).alias("day_of_week"),  # 월=1 ~ 일=7
    pl.col("ts").dt.weekday().is_in([6, 7]).alias("is_weekend"),
    pl.col("ts").dt.date().alias("ts_day"),

    # 새벽 여부 (예: 0~5시)
    pl.col("ts").dt.hour().is_between(0, 5).alias("is_dawn"),

    # TimeofDay bucket (예시: 새벽/오전/오후/밤)
    pl.when(pl.col("ts").dt.hour().is_between(0, 5)).then(pl.lit("dawn"))
      .when(pl.col("ts").dt.hour().is_between(6, 11)).then(pl.lit("morning"))
      .when(pl.col("ts").dt.hour().is_between(12, 17)).then(pl.lit("afternoon"))
      .otherwise(pl.lit("evening"))
      .alias("timeofday_bucket"),

]).with_columns([
    # 추가: Hour 주기성(sin/cos)
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).sin().alias("hour_sin"),
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).cos().alias("hour_cos"),

    # 추가: 요일을 범주형으로(순서 제거)
    pl.col("day_of_week").cast(pl.Utf8).cast(pl.Categorical).alias("dow_cat"),
])

In [ ]:
# -------------------------
# (C) Amount 파생 (float32 + log)
# -------------------------
q_feat = q_feat.with_columns([
    # float32 캐스팅
    pl.col("Amount_Paid_USD").cast(pl.Float32),
    pl.col("Amount_Received_USD").cast(pl.Float32),

    # log1p
    pl.col("Amount_Paid_USD").log1p().alias("log_amount_paid_usd"),
    pl.col("Amount_Received_USD").log1p().alias("log_amount_received_usd"),

    # Round number (1000 단위)
    (pl.col("Amount_Paid_USD") % 1000 == 0).alias("is_round_1000_paid"),
    (pl.col("Amount_Received_USD") % 1000 == 0).alias("is_round_1000_received"),

    # 더 강한 라운드 (10000 단위)
    (pl.col("Amount_Paid_USD") % 10000 == 0).alias("is_round_10000_paid"),
])

In [ ]:
# -------------------------
# (D) Payment Format 원핫 + 파생
# -------------------------
# 원핫(각 format별 bool)
q_feat = q_feat.with_columns([
    (pl.col("Payment Format") == "ACH").alias("is_ach"),
    (pl.col("Payment Format") == "Cheque").alias("is_cheque"),
    (pl.col("Payment Format") == "Bitcoin").alias("is_bitcoin_fmt"),
    (pl.col("Payment Format") == "Cash").alias("is_cash"),
    (pl.col("Payment Format") == "Credit Card").alias("is_credit_card"),
    (pl.col("Payment Format") == "Wire").alias("is_wire"),
    (pl.col("Payment Format") == "Reinvestment").alias("is_reinvestment"),
])

# Crypto 여부 (format이 Bitcoin이거나 currency가 Bitcoin이면 True)
q_feat = q_feat.with_columns([
    ((pl.col("Payment Format") == "Bitcoin") | (pl.col("Payment Currency") == "Bitcoin"))
      .alias("is_crypto_transfer"),

    # High value ACH (>= 1,000,000 USD & ACH)
    ((pl.col("Payment Format") == "ACH") & (pl.col("Amount_Paid_USD") >= 1_000_000))
      .alias("is_high_value_ach"),
])

# Payment_Method_Risk (format 기반 가중치)
q_feat = q_feat.with_columns([
    pl.col("Payment Format")
      .replace(PAYMENT_RISK_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("payment_method_risk"),
])

# format × amount interaction (v1용)
q_feat = q_feat.with_columns([
    (pl.col("payment_method_risk") * pl.col("log_amount_paid_usd")).alias("risk_x_log_paid"),
    (pl.col("is_ach").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("ach_x_log_paid"),
    (pl.col("is_bitcoin_fmt").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("btc_x_log_paid"),
])

In [ ]:
# -------------------------
# (E) Entity type 파싱 + 점수
# -------------------------
# "Corporation #26522" → "Corporation"
q_feat = q_feat.with_columns([
    pl.col("Sender_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")   # "#숫자" 이후 제거
      .alias("sender_entity_type"),

    pl.col("Receiver_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")
      .alias("receiver_entity_type"),
])

# 타입 점수
q_feat = q_feat.with_columns([
    pl.col("sender_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("sender_entity_type_score"),

    pl.col("receiver_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("receiver_entity_type_score"),
])


In [ ]:
# -------------------------
# (F) bank 관련
# -------------------------
q_feat = q_feat.with_columns([
    pl.col("Sender_Bank_Name").is_in(HIGH_RISK_SENDER_BANKS).alias("high_risk_sender_bank_flag"),
    pl.col("Receiver_Bank_Name").is_in(HIGH_RISK_RECEIVER_BANKS).alias("high_risk_receiver_bank_flag"),
])

q_feat = q_feat.with_columns([
    pl.col("From Bank").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("To Bank").cast(pl.Utf8).cast(pl.Categorical),
])

q_feat = q_feat.with_columns([
    pl.col("Sender_Bank_Branch_ID").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("Receiver_Bank_Branch_ID").cast(pl.Utf8).cast(pl.Categorical),
])

In [ ]:
# 확인하고 싶은 핵심 피처만 뽑아서 head 5
inspect_cols = [
    # timestamp 관련
    "Timestamp", "ts", "hour", "day_of_week", "is_weekend", "is_dawn", "timeofday_bucket",

    # amount 관련
    "Amount_Paid_USD", "log_amount_paid_usd",
    "Amount_Received_USD", "log_amount_received_usd",
    "is_round_1000_paid",

    # payment format
    "Payment Format",
    "is_ach", "is_bitcoin_fmt", "is_crypto_transfer",
    "is_high_value_ach", "payment_method_risk", "risk_x_log_paid",

    # entity / bank
    "Sender_Entity", "sender_entity_type", "sender_entity_type_score",
    "Sender_Bank_Name", "high_risk_sender_bank_flag",
    "Sender_Bank_Branch_ID", "Receiver_Bank_Branch_ID",
    # "Bank_Tier", "Bank_Type", "Is_Intra_Bank",

    # Nationality
    "From Country", "To Country", "IsCrossBorder", "IsFromFATF", "IsToFATF",

    # label / split
    "Is Laundering", "split"
]

q_feat.select(inspect_cols).head(5).collect()

Timestamp,ts,hour,day_of_week,is_weekend,is_dawn,timeofday_bucket,Amount_Paid_USD,log_amount_paid_usd,Amount_Received_USD,log_amount_received_usd,is_round_1000_paid,Payment Format,is_ach,is_bitcoin_fmt,is_crypto_transfer,is_high_value_ach,payment_method_risk,risk_x_log_paid,Sender_Entity,sender_entity_type,sender_entity_type_score,Sender_Bank_Name,high_risk_sender_bank_flag,Sender_Bank_Branch_ID,Receiver_Bank_Branch_ID,From Country,To Country,IsCrossBorder,IsFromFATF,IsToFATF,Is Laundering,split
str,datetime[μs],i16,i8,bool,bool,str,f32,f64,f32,f64,bool,str,bool,bool,bool,bool,f32,f64,str,str,f32,str,bool,cat,cat,str,str,u8,u8,u8,i64,str
"""2022/09/01 00:00""",2022-09-01 00:00:00,0,4,false,true,"""dawn""",1228.877075,7.11467,1228.877075,7.11467,false,"""Reinvestment""",false,false,false,false,1.0,7.11467,"""Partnership #65364""","""Partnership""",1.0,"""Spain Bank #492""",false,"""493""","""493""","""Spain""","""Spain""",0,1,1,0,"""train"""
"""2022/09/01 00:00""",2022-09-01 00:00:00,0,4,false,true,"""dawn""",24.25388,3.22898,24.25388,3.22898,false,"""Reinvestment""",false,false,false,false,1.0,3.22898,"""Corporation #139903""","""Corporation""",2.0,"""Saudi Arabia Bank #35""",false,"""36""","""36""","""Saudi Arabia""","""Saudi Arabia""",0,1,1,0,"""train"""
"""2022/09/01 00:00""",2022-09-01 00:00:00,0,4,false,true,"""dawn""",3801.102783,8.24331,3801.102783,8.24331,false,"""Reinvestment""",false,false,false,false,1.0,8.24331,"""Sole Proprietorship #145159""","""Sole Proprietorship""",1.5,"""Saudi Arabia Bank #35""",false,"""36""","""36""","""Saudi Arabia""","""Saudi Arabia""",0,1,1,0,"""train"""
"""2022/09/01 00:00""",2022-09-01 00:00:00,0,4,false,true,"""dawn""",59416.894531,10.992351,59416.894531,10.992351,false,"""Reinvestment""",false,false,false,false,1.0,10.992351,"""Partnership #2387""","""Partnership""",1.0,"""Saudi Arabia Bank #35""",false,"""36""","""36""","""Saudi Arabia""","""Saudi Arabia""",0,1,1,0,"""train"""
"""2022/09/01 00:00""",2022-09-01 00:00:00,0,4,false,true,"""dawn""",8.47742,2.248912,8.47742,2.248912,false,"""Reinvestment""",false,false,false,false,1.0,2.248912,"""Partnership #16370""","""Partnership""",1.0,"""Saudi Arabia Bank #35""",false,"""36""","""36""","""Saudi Arabia""","""Saudi Arabia""",0,1,1,0,"""train"""


# 8-2) groupby 필요한 피처 엔지니어링

In [ ]:
# ---------------------------------------------------------
# 0) 준비: _one 만들고, 정렬 (rolling_*_by + over는 정렬 중요)
# ---------------------------------------------------------
q_feat = q_feat.with_columns(pl.lit(1).cast(pl.Int32).alias("_one"))

# sender/receiver 둘 다 만들 거라서, 최소 ts 정렬 + 계정별 정렬
# (sender 만들 땐 From Account 기준, receiver 만들 땐 To Account 기준 정렬이 이상적)
# 여기서는 한 번에 가려고 1) From Account 2) To Account 3) ts 로 정렬
q_feat = q_feat.sort(["From Account", "To Account", "ts"])


# ---------------------------------------------------------
# 1) Sender(From Account) - 1h / 24h
# ---------------------------------------------------------
q_feat = q_feat.with_columns([
    # 1h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_cnt_1h_past"),

    # 1h: Amount_Paid_USD sum/mean/max/std
    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_sum_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_mean_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_max_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_max_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_std_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_std_paid_1h_past"),

    # 1h: Amount_Received_USD sum/mean
    pl.col("Amount_Received_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_sum_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_mean_recv_1h_past"),

    # 24h: count, paid sum
    pl.col("_one")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("From Account")
      .alias("s_cnt_24h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("From Account")
      .alias("s_sum_paid_24h_past"),
])

# sender net (1h)
q_feat = q_feat.with_columns(
    (pl.col("s_sum_recv_1h_past") - pl.col("s_sum_paid_1h_past")).alias("s_net_1h_past")
)


# ---------------------------------------------------------
# 2) Receiver(To Account) - 1h / 24h
# ---------------------------------------------------------
# receiver 쪽은 To Account 기준으로 rolling이므로,
# 엄밀히는 q_feat = q_feat.sort(["To Account","ts"])가 가장 안전함.
# 이미 위에서 ["From Account","To Account","ts"]로 정렬했지만,
# receiver 안정성 위해 한 번 더 정렬 권장:
q_feat = q_feat.sort(["To Account", "ts"])

q_feat = q_feat.with_columns([
    # 1h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_cnt_1h_past"),

    # 1h: Amount_Received_USD sum/mean/max/std
    pl.col("Amount_Received_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_sum_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_mean_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_max_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_max_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_std_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_std_recv_1h_past"),

    # 1h: Amount_Paid_USD sum/mean
    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_sum_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_mean_paid_1h_past"),

    # 24h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("To Account")
      .alias("r_cnt_24h_past"),
])

# receiver net (1h)
q_feat = q_feat.with_columns(
    (pl.col("r_sum_recv_1h_past") - pl.col("r_sum_paid_1h_past")).alias("r_net_1h_past")
)

# ---------------------------------------------------------
# 3) TODO: n_unique (s_nuniq_to_24h_past / r_nuniq_from_24h_past)
# ---------------------------------------------------------
# rolling_n_unique_by가 없어서, 이 2개는 "rolling_*_by"만으로는 바로 못 만들고,
# 다른 방식(예: groupby_dynamic + join, 또는 window id로 explode/agg 등) 필요.
# 일단은 여기서는 제외.


In [ ]:
q_feat.select([
    "ts", "From Account", "To Account",
    "s_cnt_1h_past", "s_sum_paid_1h_past", # "s_nuniq_to_24h_past",
    "r_cnt_1h_past", "r_sum_recv_1h_past" # "r_nuniq_from_24h_past"
]).head(10).collect()

ts,From Account,To Account,s_cnt_1h_past,s_sum_paid_1h_past,r_cnt_1h_past,r_sum_recv_1h_past
datetime[μs],str,str,i32,f32,i32,f32
2022-09-01 00:03:00,"""83838C730""","""100428660""",null,null,null,null
2022-09-01 00:04:00,"""82BA3A7E0""","""100428660""",null,null,1,0.17
2022-09-01 00:04:00,"""84F748F80""","""100428660""",null,null,4,0.43
2022-09-01 00:04:00,"""84724CBF0""","""100428660""",null,null,4,0.43
2022-09-01 00:12:00,"""80468B920""","""100428660""",null,null,4,0.43
2022-09-01 00:13:00,"""844BCAFD0""","""100428660""",null,null,5,0.46
2022-09-01 00:14:00,"""8346DDE50""","""100428660""",null,null,6,0.47
2022-09-01 00:17:00,"""84972D1C0""","""100428660""",null,null,7,0.48
2022-09-01 00:17:00,"""82D243C70""","""100428660""",null,null,9,0.53


In [ ]:
q_feat.select(pl.col("ts").is_null().sum()).collect()

ts
u32
0


In [ ]:
# q_feat.select(pl.col("ts").dtype).collect()

In [ ]:
q_feat.schema["ts"]

Datetime(time_unit='us', time_zone=None)

# 9) 모델에 안 넣을 컬럼 drop    

In [ ]:
DROP_COLS_V1 = [
    # raw time / ids
    "Timestamp", "bucket_ts_str",
    # "ts", "bucket_ts",
    "hour", "day_of_week",

    # account / bank / entity identifiers
    # "From Account", "To Account",
    # "From Bank", "To Bank",
    # "Sender_Bank_Name", "Receiver_Bank_Name",
    # "Sender_Entity", "Receiver_Entity", # 얘를 드랍해야 하는 거 아니던가...
    # "Sender_Bank_Branch_ID", "Receiver_Bank_Branch_ID",

    # intermediate entity cols
    # "sender_entity_type", "receiver_entity_type", # 얘를 왜 드랍 하더라,.,,,

    # raw amount columns
    "Amount Paid", "Amount Received",
    # "Amount_Paid_USD", "Amount_Received_USD",

    # currency
    # "Receiving Currency", "Payment Currency"

    "sender_node", "receiver_node",

    "_one",
]

q_model = q_feat.drop([c for c in DROP_COLS_V1 if c in q_feat.columns])

In [ ]:
# drop 전 51
print(len(q_model.columns))
print(q_model.columns)

74
['From Bank', 'From Account', 'To Bank', 'To Account', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Is Laundering', 'Amount_Received_USD', 'Amount_Paid_USD', 'Sender_Entity', 'Sender_Bank_Name', 'Receiver_Entity', 'Receiver_Bank_Name', 'Pattern_Type', 'Pattern_Detail', 'From Country', 'To Country', 'IsCrossBorder', 'IsFromFATF', 'IsToFATF', 'Sender_Bank_Branch_ID', 'Receiver_Bank_Branch_ID', 'ts', 'split', 'bucket_ts', 'is_weekend', 'ts_day', 'is_dawn', 'timeofday_bucket', 'hour_sin', 'hour_cos', 'dow_cat', 'log_amount_paid_usd', 'log_amount_received_usd', 'is_round_1000_paid', 'is_round_1000_received', 'is_round_10000_paid', 'is_ach', 'is_cheque', 'is_bitcoin_fmt', 'is_cash', 'is_credit_card', 'is_wire', 'is_reinvestment', 'is_crypto_transfer', 'is_high_value_ach', 'payment_method_risk', 'risk_x_log_paid', 'ach_x_log_paid', 'btc_x_log_paid', 'sender_entity_type_score', 'receiver_entity_type_score', 'high_risk_sender_bank_flag', 'high_risk_receiver_bank_flag', 's_cnt

In [ ]:
FLOAT64_COLS = [
    "log_amount_paid_usd",
    "log_amount_received_usd",
    "risk_x_log_paid",
    "ach_x_log_paid",
    "btc_x_log_paid",
]

q_model = q_model.with_columns([
    pl.col(c).cast(pl.Float32) for c in FLOAT64_COLS
])

In [ ]:
q_model.schema

Schema([('From Bank', Categorical),
        ('From Account', String),
        ('To Bank', Categorical),
        ('To Account', String),
        ('Receiving Currency', String),
        ('Payment Currency', String),
        ('Payment Format', String),
        ('Is Laundering', Int64),
        ('Amount_Received_USD', Float32),
        ('Amount_Paid_USD', Float32),
        ('Sender_Entity', String),
        ('Sender_Bank_Name', String),
        ('Receiver_Entity', String),
        ('Receiver_Bank_Name', String),
        ('Pattern_Type', String),
        ('Pattern_Detail', String),
        ('From Country', String),
        ('To Country', String),
        ('IsCrossBorder', UInt8),
        ('IsFromFATF', UInt8),
        ('IsToFATF', UInt8),
        ('Sender_Bank_Branch_ID', Categorical),
        ('Receiver_Bank_Branch_ID', Categorical),
        ('ts', Datetime(time_unit='us', time_zone=None)),
        ('split', String),
        ('bucket_ts', Datetime(time_unit='us', time_zone=None)),
        

In [ ]:
#“이 컬럼이 numeric / categorical인지” 빠르게 나누기
schema = q_model.schema

numeric_cols = [
    c for c, d in schema.items()
    if d in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.Float32, pl.Float64,
        pl.Boolean,   # ✅ 추가
    )
]

categorical_cols = [
    c for c, d in schema.items()
    if d in (pl.Utf8, pl.Categorical)
]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['Is Laundering', 'Amount_Received_USD', 'Amount_Paid_USD', 'is_weekend', 'is_dawn', 'hour_sin', 'hour_cos', 'log_amount_paid_usd', 'log_amount_received_usd', 'is_round_1000_paid', 'is_round_1000_received', 'is_round_10000_paid', 'is_ach', 'is_cheque', 'is_bitcoin_fmt', 'is_cash', 'is_credit_card', 'is_wire', 'is_reinvestment', 'is_crypto_transfer', 'is_high_value_ach', 'payment_method_risk', 'risk_x_log_paid', 'ach_x_log_paid', 'btc_x_log_paid', 'sender_entity_type_score', 'receiver_entity_type_score', 'high_risk_sender_bank_flag', 'high_risk_receiver_bank_flag', 's_cnt_1h_past', 's_sum_paid_1h_past', 's_mean_paid_1h_past', 's_max_paid_1h_past', 's_std_paid_1h_past', 's_sum_recv_1h_past', 's_mean_recv_1h_past', 's_cnt_24h_past', 's_sum_paid_24h_past', 's_net_1h_past', 'r_cnt_1h_past', 'r_sum_recv_1h_past', 'r_mean_recv_1h_past', 'r_max_recv_1h_past', 'r_std_recv_1h_past', 'r_sum_paid_1h_past', 'r_mean_paid_1h_past', 'r_cnt_24h_past', 'r_net_1h_past']
Categorical: ['From Bank'

# 10) 마지막에 pandas로 collect

In [ ]:
split_counts = (
    q_feat.group_by("split")
          .agg(pl.count().alias("rows"))
          .collect(engine="streaming")
)
print(split_counts)

shape: (3, 2)
┌───────┬──────────┐
│ split ┆ rows     │
│ ---   ┆ ---      │
│ str   ┆ u32      │
╞═══════╪══════════╡
│ test  ┆ 6378958  │
│ train ┆ 19139731 │
│ val   ┆ 6379980  │
└───────┴──────────┘


In [ ]:
label_dist = (
    q_feat.group_by(["split", "Is Laundering"])
          .agg(pl.count().alias("rows"))
          .sort(["split", "Is Laundering"])
          .collect(engine="streaming")
)
print(label_dist)

shape: (6, 3)
┌───────┬───────────────┬──────────┐
│ split ┆ Is Laundering ┆ rows     │
│ ---   ┆ ---           ┆ ---      │
│ str   ┆ i64           ┆ u32      │
╞═══════╪═══════════════╪══════════╡
│ test  ┆ 0             ┆ 6368318  │
│ test  ┆ 1             ┆ 10640    │
│ train ┆ 0             ┆ 19124195 │
│ train ┆ 1             ┆ 15536    │
│ val   ┆ 0             ┆ 6370926  │
│ val   ┆ 1             ┆ 9054     │
└───────┴───────────────┴──────────┘


In [ ]:
# =========================================================
# sklearn로 넘기기 위한 "컬럼만" 선택 후 collect
# =========================================================
label_col = "Is Laundering"
META_COLS = ["ts_day", "From Account", "To Account", "ts", "bucket_ts", "Pattern_Type", "Pattern_Detail"]  # KPI용 메타

feature_cols = [
    c for c in q_model.columns
    if c not in (["split", label_col] + META_COLS)
]

q_train = (
    q_model.filter(pl.col("split") == "train")
          .select(feature_cols + [label_col] + META_COLS)
)
train_df = q_train.collect(engine="streaming").to_pandas()

q_val = (
    q_model.filter(pl.col("split") == "val")
          .select(feature_cols + [label_col] + META_COLS)
)
val_df   = q_val.collect(engine="streaming").to_pandas()

q_test = (
    q_model.filter(pl.col("split") == "test")
          .select(feature_cols + [label_col] + META_COLS)
)
test_df  = q_test.collect(engine="streaming").to_pandas()

print(train_df.shape, val_df.shape, test_df.shape)

(19139731, 73) (6379980, 73) (6378958, 73)


# 11) Top-k 로직(하루 기준/누적 기준)

In [ ]:
# Top-k 로직 교체용 함수:“k개 적발하려면 몇 건을 봐야 하는지”를 계산하는 함수
def workload_to_find_k_positives(y_true, y_score, k_pos):
    """
    목표: true positive를 k_pos개 '찾기' 위해
         상위 점수부터 몇 건(N)을 조사해야 하는지 계산

    Returns:
      N_required: 필요한 조사 건수
      precision:  k_found / N_required
      recall:     k_found / total_pos
      f1:         f1 (top-N을 양성으로 가정한 경우)
      k_found:    실제로 찾은 TP 수 (total_pos < k_pos면 total_pos)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    total_pos = y_true.sum()
    if total_pos == 0:
        return 0, 0.0, 0.0, 0.0, 0

    # 점수 내림차순 정렬
    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    # 누적 TP
    cum_tp = np.cumsum(y_sorted)

    target = min(k_pos, int(total_pos))

    # cum_tp >= target 되는 첫 index
    idx = np.searchsorted(cum_tp, target, side="left")
    N_required = int(idx + 1)  # index -> count

    k_found = int(cum_tp[idx])  # 보통 target과 같음(동점/중복 없으면)

    precision = k_found / (N_required + 1e-12)
    recall = k_found / (total_pos + 1e-12)

    # top-N을 양성으로 보면: TP=k_found, FP=N-TP, FN=total_pos-TP
    fp = N_required - k_found
    fn = total_pos - k_found
    f1 = (2 * k_found) / (2 * k_found + fp + fn + 1e-12)

    return N_required, precision, recall, f1, k_found

In [ ]:
def per_day_workload_summary(df, day_col, label_col, score_col, k_pos_list=(30,50,100,200)):
    """
    df: pandas DataFrame with [day_col, label_col, score_col]
    day별로 'k_pos개 적발하기 위해 필요한 조사량 N' 계산 후
    k_pos마다 daily 분포 + mean/median/p90 요약을 반환
    """
    out_rows = []
    for day, g in df.groupby(day_col):
        y = g[label_col].astype(int).values
        s = g[score_col].values
        total_pos = int(y.sum())
        n = len(g)

        for k_pos in k_pos_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y, s, k_pos)
            out_rows.append({
                "day": day,
                "n_rows": n,
                "pos": total_pos,
                "k_pos": k_pos,
                "N_required": N_req,
                "precision_at_Nk": p,
                "recall_at_Nk": r,
                "f1_at_Nk": f1,
                "found_tp": found,
                "target_k": min(k_pos, total_pos),
                "hit_target": (found >= min(k_pos, total_pos)) and (total_pos > 0)
            })

    daily = pd.DataFrame(out_rows)

    # day에 pos=0이면 N_required=0이 되고 의미가 약하니,
    # 운영 KPI로는 "pos>0인 day"만 요약하는게 보통 더 깔끔함
    daily_pos = daily[daily["pos"] > 0].copy()

    def summarize(sub):
        # N_required는 "작을수록 좋음"
        return pd.Series({
            "days": sub["day"].nunique(),
            "mean_N_required": sub["N_required"].mean(),
            "median_N_required": sub["N_required"].median(),
            "p90_N_required": sub["N_required"].quantile(0.90),
            "mean_precision": sub["precision_at_Nk"].mean(),
            "median_precision": sub["precision_at_Nk"].median(),
            "p90_precision": sub["precision_at_Nk"].quantile(0.90),
        })

    summary = (
        daily_pos
        .groupby("k_pos", as_index=False)
        .apply(summarize)
        .reset_index(drop=True)
    )

    return daily, summary

In [ ]:
def log_per_day_kpi_to_wandb(split_name, df_raw_with_day, y_true, y_score, k_list=(30,50,100,200)):
    """
    split_name: "val" or "test"
    df_raw_with_day: pandas DF that includes column "ts_day" aligned with y_true/y_score index
    y_true/y_score: numpy-like aligned arrays
    """
    tmp = pd.DataFrame({
        "ts_day": df_raw_with_day["ts_day"].values,
        "label": np.asarray(y_true).astype(int),
        "score": np.asarray(y_score),
    })

    daily, summary = per_day_workload_summary(
        tmp,
        day_col="ts_day",
        label_col="label",
        score_col="score",
        k_pos_list=k_list
    )

    # (A) 요약 로그: k별 mean/median/p90
    for _, row in summary.iterrows():
        k = int(row["k_pos"])
        wandb.log({
            f"{split_name}_perday_find_{k}_mean_N": float(row["mean_N_required"]),
            f"{split_name}_perday_find_{k}_median_N": float(row["median_N_required"]),
            f"{split_name}_perday_find_{k}_p90_N": float(row["p90_N_required"]),
            f"{split_name}_perday_find_{k}_mean_precision": float(row["mean_precision"]),
            f"{split_name}_perday_find_{k}_median_precision": float(row["median_precision"]),
            f"{split_name}_perday_find_{k}_p90_precision": float(row["p90_precision"]),
            f"{split_name}_perday_days_with_pos_{k}": int(row["days"]),
        })

    # (B) 분포 확인용 테이블(너무 크면 상위 일부만)
    # day*k 만큼 행이 생김 -> 기간 짧으면 OK, 길면 샘플링/요약만 남기기
    table = wandb.Table(dataframe=daily.head(5000))
    wandb.log({f"{split_name}_perday_workload_table": table})

In [ ]:
def topN_distinct_account_metrics(df_meta, y_true, y_score, topN,
                                  account_cols=("From Account","To Account"),
                                  score_agg="max"):
    """
    전체 기간에서 score 내림차순 topN '거래'를 뽑고,
    그 안에서 account_cols(From/To) 기준 distinct 계좌로 묶어 평가.

    label: 계좌가 topN 거래 중 laundering 거래가 1번이라도 있으면 1 (max)
    score: 계좌 내 score 집계 (max 추천)
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # topN 거래 선택
    tmp = tmp.sort_values("score_trx", ascending=False).head(int(min(topN, len(tmp))))

    # 거래 -> 계좌 집계
    agg = aggregate_to_account_level(
        df_meta=tmp, y_true=tmp["label_trx"].values, y_score=tmp["score_trx"].values,
        account_cols=account_cols, score_agg=score_agg
    )

    y_acc = agg["label"].values
    tp = int((y_acc == 1).sum())
    fp = int((y_acc == 0).sum())
    precision = tp / (tp + fp + 1e-12)

    return {
        "topN_trx": int(min(topN, len(tmp))),
        "distinct_accounts": int(len(agg)),
        "tp_accounts": tp,
        "fp_accounts": fp,
        "precision_accounts": float(precision),
    }

In [ ]:
def score_bin_distribution(y_true, y_score, n_bins=10, score_scale=1000):
    """
    score를 0~score_scale로 스케일링한 뒤,
    bin별 label(0/1) 건수 집계.
    항상 normal_cnt / laundering_cnt 컬럼이 존재하도록 보정.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    s_scaled = np.clip(s * score_scale, 0, score_scale)
    bins = np.linspace(0, score_scale, n_bins + 1)

    # 0..n_bins-1
    bin_idx = np.digitize(s_scaled, bins, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    df = pd.DataFrame({"bin": bin_idx, "label": y})

    # label별 count (항상 0/1 둘 다 컬럼을 만들도록 reindex)
    cnt = (
        df.groupby(["bin", "label"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0)   # ✅ 핵심
          .rename(columns={0: "normal_cnt", 1: "laundering_cnt"})
          .reset_index()
    )

    # bin_range 붙이기
    cnt["bin_range"] = cnt["bin"].apply(lambda i: f"{int(bins[i])}-{int(bins[i+1])}")

    # 보기 좋게 정렬
    cnt = cnt[["bin", "bin_range", "normal_cnt", "laundering_cnt"]]
    return cnt

In [ ]:
def aggregate_to_account_level(df_meta, y_true, y_score,
                               account_cols=("From Account",),
                               score_agg="max"):
    """
    거래 단위 (y_true, y_score)를 계좌 단위로 집계.
    account_cols: ("From Account",) or ("To Account",) or ("From Account","To Account")
    score_agg: "max" (추천) or "mean"
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    # (A) 거래 메타 + label/score 결합
    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # (B) From/To 둘 다 쓰는 경우: long 형태로 펼치기
    if len(account_cols) == 2:
        a1, a2 = account_cols
        tmp1 = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})
        tmp2 = tmp[[a2, "label_trx", "score_trx"]].rename(columns={a2: "account"})
        tmp_long = pd.concat([tmp1, tmp2], axis=0, ignore_index=True)
    else:
        a1 = account_cols[0]
        tmp_long = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})

    # 결측 계좌 방어
    tmp_long = tmp_long.dropna(subset=["account"])

    # (C) 계좌 단위 집계
    if score_agg == "mean":
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "mean"),
            n_hits=("score_trx", "size"),
        ).reset_index()
    else:  # "max"
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "max"),
            n_hits=("score_trx", "size"),
        ).reset_index()

    return agg  # columns: account, label, score, n_hits

In [ ]:
def log_per_day_account_kpi_to_wandb(split_name, df_meta, y_true, y_score,
                                     k_list=(30,50,100,200),
                                     account_cols=("From Account","To Account"),
                                     score_agg="max"):
    """
    하루마다 (거래->계좌 집계) 후 workload_to_find_k_positives를 계좌단위로 계산.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[["ts_day"] + list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    rows = []
    for day, g in tmp.groupby("ts_day"):
        agg = aggregate_to_account_level(
            df_meta=g, y_true=g["label_trx"].values, y_score=g["score_trx"].values,
            account_cols=account_cols, score_agg=score_agg
        )
        y_acc = agg["label"].values
        s_acc = agg["score"].values

        for k_pos in k_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y_acc, s_acc, k_pos)
            rows.append({
                "ts_day": day,
                "k_pos": k_pos,
                "n_accounts": len(agg),
                "pos_accounts": int(y_acc.sum()),
                "N_required": N_req,
                "precision": p,
                "recall": r,
                "f1": f1,
                "found_tp": found
            })

    daily = pd.DataFrame(rows)
    daily_pos = daily[daily["pos_accounts"] > 0].copy()

    # 요약 로그
    if len(daily_pos) > 0:
        summ = daily_pos.groupby("k_pos").agg(
            days=("ts_day", "nunique"),
            mean_N=("N_required", "mean"),
            median_N=("N_required", "median"),
            p90_N=("N_required", lambda x: x.quantile(0.9)),
            mean_precision=("precision", "mean"),
        ).reset_index()

        for _, r in summ.iterrows():
            k = int(r["k_pos"])
            wandb.log({
                f"{split_name}_acc_perday_find_{k}_mean_N": float(r["mean_N"]),
                f"{split_name}_acc_perday_find_{k}_median_N": float(r["median_N"]),
                f"{split_name}_acc_perday_find_{k}_p90_N": float(r["p90_N"]),
                f"{split_name}_acc_perday_find_{k}_mean_precision": float(r["mean_precision"]),
                f"{split_name}_acc_perday_days_with_pos_{k}": int(r["days"]),
            })

    wandb.log({f"{split_name}_acc_perday_table": wandb.Table(dataframe=daily.head(5000))})

In [ ]:
def get_pattern_analysis_tables(df, prob, thr):
    """
    Pandas 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환
    """

    # 1. 실제 세탁 거래(1)만 추출하여 분석용 DF 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": df["Is Laundering"],
        "is_detected": (prob >= thr).astype(int)
    })
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return wandb.Table(columns=["Pattern_Type", "Recall_Pct"]), wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Recall_Pct"])

    # 2. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 잘 못잡는 순서 정렬

    # 3. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [ ]:
def get_pattern_analysis_tables_find_k(df, prob, k_pos_target):
    """
    Find-K 로직 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환.
    thr 대신 k_pos_target(예: 3000)을 사용하여 상위 N개 조사 범위를 결정.
    """
    y_true = df["Is Laundering"].astype(int).values
    y_score = np.asarray(prob)

    # 1. 외부 함수를 사용하여 k_pos_target개를 찾기 위한 N_required 계산
    # (이미 정의하신 workload_to_find_k_positives 함수를 활용합니다)
    N_req, _, _, _, _ = workload_to_find_k_positives(y_true, y_score, k_pos_target)

    # 2. 분석용 데이터프레임 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": y_true,
        "score": y_score
    })

    # 점수 순 정렬 후 상위 N_req개만 'is_detected'로 표시
    analysis = analysis.sort_values("score", ascending=False).reset_index(drop=True)
    analysis["is_detected"] = 0
    analysis.loc[:N_req-1, "is_detected"] = 1

    # 3. 실제 세탁 거래(1)만 추출하여 패턴 분석 진행
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return (wandb.Table(columns=["Pattern_Type", "Total_Actual", "Detected_Count", "Recall_Pct"]),
                wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Total_Actual", "Detected_Count", "Recall_Pct"]))

    # 4. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 검출률 낮은 순 정렬

    # 5. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()

    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [ ]:
def log_detailed_error_analysis(split_name, df, y_true, y_score, threshold, top_features):
    """
    정탐(TP), 오탐(FP), 미탐(FN) 그룹별로 피처 분포와 일자별 추이를 분석하여 WandB에 로깅
    """
    analysis_df = df.copy()
    analysis_df['prob'] = y_score
    analysis_df['pred'] = (y_score >= threshold).astype(int)
    analysis_df['label'] = y_true.astype(int)

    # TP, FP, FN, TN 분류
    def categorize(row):
        if row['label'] == 1 and row['pred'] == 1: return 'TP (정탐)'
        if row['label'] == 0 and row['pred'] == 1: return 'FP (오탐)'
        if row['label'] == 1 and row['pred'] == 0: return 'FN (미탐)'
        return 'TN'

    analysis_df['error_cat'] = analysis_df.apply(categorize, axis=1)

    # (1) 일자별 에러 분포 (정탐/오탐/미탐 건수 추이)
    # 2022/09/14 ~ 2022/09/28 범위의 트렌드 파악용
    error_by_day = analysis_df.groupby(['ts_day', 'error_cat']).size().unstack(fill_value=0).reset_index()

    thr_tag = f"thr{threshold:.2f}"

    wandb.log({f"{split_name}_{thr_tag}_daily_error_trend": wandb.Table(dataframe=error_by_day)})

    # (2) 피처별 에러 특성 분석 (어떤 피처가 오탐/미탐에 영향을 주는가?)
    # 중요도 상위 피처(top_features)에 대해 각 그룹별 평균 또는 빈도 계산
    feat_stats = []
    for cat in ['TP (정탐)', 'FP (오탐)', 'FN (미탐)']:
        sub = analysis_df[analysis_df['error_cat'] == cat]
        if sub.empty: continue

        stat_row = {'category': cat, 'count': len(sub)}
        for f in top_features:
            if f not in sub.columns: continue
            val = sub[f]
            # 수치형 피처: 평균값 비교
            if pd.api.types.is_numeric_dtype(val):
                stat_row[f"{f}_mean"] = round(val.mean(), 4)
            # 범주형 피처: 최빈값(Mode) 비교
            else:
                stat_row[f"{f}_top_mode"] = str(val.mode()[0]) if not val.mode().empty else "N/A"
        feat_stats.append(stat_row)

    wandb.log({f"{split_name}_{thr_tag}_feature_characteristics_by_error": wandb.Table(dataframe=pd.DataFrame(feat_stats))})

In [ ]:
def log_pattern_error_analysis(split_name, df, y_true, y_score, threshold):
    """
    미탐(FN)과 오탐(FP) 건들에 대해 일자별 패턴 분포를 집중 분석
    """
    df_err = df.copy()
    df_err['pred'] = (y_score >= threshold).astype(int)
    df_err['label'] = y_true.astype(int)

    # FN (미탐): 실제 세탁인데 놓친 것
    fn_df = df_err[(df_err['label'] == 1) & (df_err['pred'] == 0)]
    # FP (오탐): 정상인데 세탁으로 예측한 것
    fp_df = df_err[(df_err['label'] == 0) & (df_err['pred'] == 1)]

    thr_tag = f"thr{threshold:.2f}"

    # (1) 미탐(FN) 패턴 분석: 어떤 날에 어떤 패턴을 주로 놓쳤나?
    if not fn_df.empty:
        fn_pattern_day = fn_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='missed_count')
        wandb.log({f"{split_name}_{thr_tag}_missed_patterns_by_day": wandb.Table(dataframe=fn_pattern_day)})

        # 상세 패턴 미탐 (Pattern_Detail)
        fn_detail = fn_df.groupby(['Pattern_Type', 'Pattern_Detail']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_{thr_tag}_missed_patterns_detail": wandb.Table(dataframe=fn_detail)})

    # (2) 오탐(FP) 패턴 분석: 오탐된 건들은 주로 어떤 패턴(혹은 일반 거래)인가?
    if not fp_df.empty:
        # 정상 거래에 패턴 정보가 태깅되어 있는 경우(Synthetic 데이터 등) 유용함
        fp_pattern = fp_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='false_alarm_count')
        wandb.log({f"{split_name}_{thr_tag}_false_alarm_patterns_by_day": wandb.Table(dataframe=fp_pattern)})

In [ ]:
def log_find_k_error_analysis(split_name, df, y_true, y_score, k_pos_target, top_features):
    """
    find-K 로직과 호환되는 에러 분석:
    실제 Positive를 k_pos_target개 찾기 위해 필요한 상위 N개 구간을 정해 분석함.
    """
    y_true_arr = np.asarray(y_true).astype(int)
    y_score_arr = np.asarray(y_score)

    # 1. 외부함수 활용하여 필요한 조사 건수(N_req) 계산
    N_req, precision, recall, f1, k_found = workload_to_find_k_positives(y_true_arr, y_score_arr, k_pos_target)

    # 2. 분석용 DF 생성
    analysis_df = df.copy()
    analysis_df['prob'] = y_score_arr
    analysis_df['label'] = y_true_arr

    # 점수 내림차순 정렬 후 상위 N_req개에 대해 pred=1 부여
    analysis_df = analysis_df.sort_values("prob", ascending=False).reset_index(drop=True)
    analysis_df['pred'] = 0
    analysis_df.loc[:N_req-1, 'pred'] = 1  # 0부터 N_req건까지 1 부여

    # 3. TP/FP/FN 분류 (N_req 구간 기준)
    # TP: 상위 N개 중 실제 세탁 (k_found개)
    # FP: 상위 N개 중 실제 정상 (N_req - k_found개)
    # FN: 상위 N개 밖에 있는 실제 세탁 (total_pos - k_found개)
    def categorize(row):
        if row['label'] == 1 and row['pred'] == 1: return 'TP (정탐)'
        if row['label'] == 0 and row['pred'] == 1: return 'FP (오탐)'
        if row['label'] == 1 and row['pred'] == 0: return 'FN (미탐)'
        return 'TN'

    analysis_df['error_cat'] = analysis_df.apply(categorize, axis=1)

    # --- [A] 일자별 분석 (Trend) ---
    # find-K를 위해 조사한 N개 내에서의 일자별 정탐/오탐/미탐 분포
    error_by_day = analysis_df.groupby(['ts_day', 'error_cat']).size().unstack(fill_value=0).reset_index()
    wandb.log({f"{split_name}_find_{k_pos_target}_daily_error_trend": wandb.Table(dataframe=error_by_day)})

    # --- [B] 피처별 분석 (Characteristics) ---
    feat_stats = []
    for cat in ['TP (정탐)', 'FP (오탐)', 'FN (미탐)']:
        sub = analysis_df[analysis_df['error_cat'] == cat]
        if sub.empty: continue

        stat_row = {'category': cat, 'count': len(sub)}
        for f in top_features:
            if f not in sub.columns: continue
            val = sub[f]
            if pd.api.types.is_numeric_dtype(val):
                stat_row[f"{f}_mean"] = round(val.mean(), 4)
            else:
                stat_row[f"{f}_top_mode"] = str(val.mode()[0]) if not val.mode().empty else "N/A"
        feat_stats.append(stat_row)

    wandb.log({f"{split_name}_find_{k_pos_target}_feat_stats": wandb.Table(dataframe=pd.DataFrame(feat_stats))})

    # --- [C] 패턴 중심 분석 ---
    # 미탐(FN): 3000개를 잡기 위해 조사해야 하는 우선순위 밖으로 밀려난 패턴들
    fn_df = analysis_df[analysis_df['error_cat'] == 'FN (미탐)']
    if not fn_df.empty:
        fn_pattern = fn_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_by_day": wandb.Table(dataframe=fn_pattern)})

        fn_detail = fn_df.groupby(['Pattern_Type', 'Pattern_Detail']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_detail": wandb.Table(dataframe=fn_detail)})

    # 오탐(FP): 3000개를 잡는 과정에서 조사가 불가피했던 정상 패턴들
    fp_df = analysis_df[analysis_df['error_cat'] == 'FP (오탐)']
    if not fp_df.empty:
        fp_pattern = fp_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fp_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_false_alarm_patterns_by_day": wandb.Table(dataframe=fp_pattern)})

# 12-1) W&B Sweeps용 train/eval 함수 정의(xgboost)

In [ ]:
def train_eval_sweep():
    """
    W&B agent가 호출하는 sweep train function.
    - run 안에서 transformer fit은 train만
    - val/test는 transform만 (누수 방지)
    - scale_pos_weight sweep
    - val 기준으로 sweep 최적화 metric 로깅
    - PR curve / CM / Top-K는 val/test 각각 로깅
    - XGB feature importance(gain) 로깅 (변환 후 feature name)
    """
    run = wandb.init()
    config = wandb.config
    set_config(transform_output="pandas")

    # =========================
    # (0) RAW -> X/y 분리
    # =========================
    LABEL_COL = "Is Laundering"

    # train/val/test는 "이미 time split 완료된 pandas df"라고 가정
    # (run 밖에서 만들어둔 걸 그대로 씀)
    X_train_raw = train_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_train     = train_df[LABEL_COL].astype(int)

    X_val_raw   = val_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_val       = val_df[LABEL_COL].astype(int)

    X_test_raw  = test_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_test      = test_df[LABEL_COL].astype(int)

    # =========================
    # (0.5) Safety check: forbidden columns
    # =========================
    FORBIDDEN_COLS = {
        "From Account", # 계좌O 버전에서는 주석처리
        "To Account", # 계좌O 버전에서는 주석처리
        "sender_node",
        "receiver_node",
        "ts",
        "bucket_ts",
        "bucket_ts_str",
        "Timestamp",
    }

    for split_name, X in [
        ("train", X_train_raw),
        ("val",   X_val_raw),
        ("test",  X_test_raw),
    ]:
        leaked = FORBIDDEN_COLS.intersection(X.columns)
        assert len(leaked) == 0, (
            f"[LEAKAGE ERROR] {split_name} input contains forbidden columns: {leaked}"
        )

    # -------------------------
    # 0) 컬럼 그룹 정의
    # -------------------------
    NATIVE_CAT_COLS = [
        "dow_cat",
        "Receiving Currency",
        "Payment Currency",
        "Payment Format",
        "timeofday_bucket",
        "Sender_Bank_Branch_ID",
        "Receiver_Bank_Branch_ID",
    ]

    HIGH_CARD_COLS = [
        "Sender_Entity",
        "Sender_Bank_Name",
        "Receiver_Entity",
        "Receiver_Bank_Name",
        # "From Account", # 계좌O에서는 주석 해제
        # "To Account",  # 계좌O에서는 주석 해제
        "From Country",
        "To Country",
        "From Bank",
        "To Bank",
    ]

    # -------------------------
    # 1) Top-K + RARE (train-only) 사전 만들기 + 적용
    #    - 누수 방지: fit은 train에서만
    # -------------------------
    def fit_topk_maps(df_train: pd.DataFrame, cols, top_k=2000, min_count=None):
        """
        cols 각각에 대해:
        - top_k: 가장 많이 나온 카테고리 top_k만 유지
        - min_count: (선택) 빈도 min_count 미만은 RARE로
        반환: {col: set(allowed_categories)}
        """
        maps = {}
        for c in cols:
            if c not in df_train.columns:
                continue
            s = df_train[c].astype("string").fillna("__MISSING__")
            vc = s.value_counts(dropna=False)

            if min_count is not None:
                allowed = set(vc[vc >= min_count].index.astype(str))
            else:
                allowed = set(vc.head(top_k).index.astype(str))

            # MISSING은 항상 허용(해석 가능 + 안정)
            allowed.add("__MISSING__")
            maps[c] = allowed
        return maps

    def apply_topk_maps(df: pd.DataFrame, maps, rare_token="__RARE__"):
        df = df.copy()
        for c, allowed in maps.items():
            if c not in df.columns:
                continue
            s = df[c].astype("string").fillna("__MISSING__")
            df[c] = np.where(s.isin(list(allowed)), s, rare_token)
        return df

    # -------------------------
    # 2) category dtype 강제 (XGBoost native categorical용)
    # -------------------------
    def force_category(df: pd.DataFrame, cols):
        df = df.copy()
        for c in cols:
            if c in df.columns:
                df[c] = df[c].astype("category")
        return df

    # ---- (A) train 기준 top-k 사전 fit
    topk_maps = fit_topk_maps(X_train_raw, HIGH_CARD_COLS, top_k=2000, min_count=None)

    # ---- (B) train/val/test에 동일하게 적용 (누수 없음)
    X_train_raw2 = apply_topk_maps(X_train_raw, topk_maps)
    X_val_raw2   = apply_topk_maps(X_val_raw,   topk_maps)
    X_test_raw2  = apply_topk_maps(X_test_raw,  topk_maps)

    # ---- (C) native-cat + high-card(정규화된) 모두 category로 고정
    CAT_COLS_FINAL = [c for c in (NATIVE_CAT_COLS + HIGH_CARD_COLS) if c in X_train_raw2.columns]

    X_train_raw2 = force_category(X_train_raw2, CAT_COLS_FINAL)
    X_val_raw2   = force_category(X_val_raw2,   CAT_COLS_FINAL)
    X_test_raw2  = force_category(X_test_raw2,  CAT_COLS_FINAL)

    # -------------------------
    # bool 컬럼은 numeric으로 캐스팅 (median impute 안전)
    #    (너 로그에 떨어져나간 애들 대부분 bool일 가능성이 큼)
    # -------------------------
    def cast_bool_to_int(df):
        df = df.copy()
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for c in bool_cols:
            df[c] = df[c].astype("int8")
        return df

    X_train_raw2 = cast_bool_to_int(X_train_raw2)
    X_val_raw2   = cast_bool_to_int(X_val_raw2)
    X_test_raw2  = cast_bool_to_int(X_test_raw2)

    # -------------------------
    # 4) 전처리기: numeric만 impute, categorical은 passthrough로 "진짜 category" 유지
    #    + 결과를 DataFrame으로 받게 됨 (set_config 덕분)
    # -------------------------
    all_cols = X_train_raw2.columns.tolist()
    num_cols = [c for c in all_cols if c not in CAT_COLS_FINAL]
    cat_cols = [c for c in CAT_COLS_FINAL if c in all_cols]

    # 3) 혹시라도 num/cat 둘 다 아닌 "문자열 잔여 컬럼"이 있다면
    #    -> 지금은 안전하게 drop (원하면 나중에 별도 처리)
    rest_cols = [c for c in X_train_raw2.columns if (c not in num_cols and c not in cat_cols)]
    if len(rest_cols) > 0:
        print("[WARN] Dropping non-numeric non-category cols:", rest_cols)
        X_train_raw2 = X_train_raw2.drop(columns=rest_cols)
        X_val_raw2   = X_val_raw2.drop(columns=rest_cols)
        X_test_raw2  = X_test_raw2.drop(columns=rest_cols)

    # 4) transformer 구성 (num만 impute, cat은 passthrough)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
        ],
        remainder="passthrough",   # cat_cols는 그대로 뒤에 붙음
        verbose_feature_names_out=False,
    )

    X_train = transformer.fit_transform(X_train_raw2)
    X_val   = transformer.transform(X_val_raw2)
    X_test  = transformer.transform(X_test_raw2)

    # ✅ passthrough로 온 cat 컬럼이 혹시 object로 바뀌었으면 다시 category로 고정
    # (환경/버전에 따라 가끔 필요)
    for c in CAT_COLS_FINAL:
        if c in X_train.columns:
            X_train[c] = X_train[c].astype("category")
            X_val[c]   = X_val[c].astype("category")
            X_test[c]  = X_test[c].astype("category")

    # feature names
    feature_names = X_train.columns.tolist()

    # =========================
    # (DEBUG) categorical dtype 살아있는지 확인
    # =========================
    print("=== Raw2 categorical dtypes ===")
    print(X_train_raw2[cat_cols].dtypes)

    print("\n=== Transformed X_train info ===")
    print("type(X_train):", type(X_train))
    print("X_train dtype:", getattr(X_train, "dtype", None))

    # =========================
    # (A) 모델 생성 (sweep 파라미터)
    # =========================
    xgb_model = XGBClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        learning_rate=config.learning_rate,
        subsample=config.subsample,
        colsample_bytree=config.colsample_bytree,
        reg_lambda=config.reg_lambda,
        reg_alpha=config.reg_alpha,
        min_child_weight=config.min_child_weight,
        gamma=config.gamma,
        scale_pos_weight=config.scale_pos_weight,   # ✅ sweep
        tree_method="hist",
        enable_categorical=True,
        objective="binary:logistic",  # 기본
        eval_metric="aucpr",           # 학습 모니터링용
        random_state=42,
        n_jobs=-1
    )

    # =========================
    # (B) 학습
    # =========================
    xgb_model.fit(X_train, y_train)

    # =========================
    # (C) 평가 (val/test 둘 다)
    # =========================
    def safe_auc_metrics(y_true, y_score):
        if len(np.unique(y_true)) <= 1:
            return np.nan, np.nan
        return roc_auc_score(y_true, y_score), average_precision_score(y_true, y_score)

    def topk_metrics(y_true, y_score, k):
        k = min(k, len(y_true))
        idx = np.argsort(-y_score)[:k]
        y_pred = np.zeros_like(y_true)
        y_pred[idx] = 1
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f = f1_score(y_true, y_pred, zero_division=0)
        return p, r, f

    def threshold_at_recall(y_true, y_score, desired_recall=0.90):
        prec, rec, thr = precision_recall_curve(y_true, y_score)
        if len(thr) == 0:
            return 0.5, prec, rec, thr
        closest_i = np.argmin(np.abs(rec[1:] - desired_recall))
        return float(thr[closest_i]), prec, rec, thr

    # -------------------------
    # (C-1) val / test 확률 예측
    # -------------------------
    prob_val  = xgb_model.predict_proba(X_val)[:, 1]
    prob_test = xgb_model.predict_proba(X_test)[:, 1]

    val_roc_auc,  val_auprc  = safe_auc_metrics(y_val, prob_val)
    test_roc_auc, test_auprc = safe_auc_metrics(y_test, prob_test)

    # ✅ sweep 최적화 metric은 val 기준으로 통일
    wandb.log({
        "val_roc_auc":  val_roc_auc,
        "val_auprc":    val_auprc,
        "test_roc_auc": test_roc_auc,
        "test_auprc":   test_auprc,
        "n_train": len(y_train),
        "pos_train": int(np.sum(y_train)),
        "pos_val": int(np.sum(y_val)),
        "pos_test": int(np.sum(y_test)),
    })

    # -------------------------
    # (C-2) PR Curve 로깅 (val/test 각각)
    # -------------------------
    # PR curve 로깅 (val/test)
    prec_v, rec_v, _ = precision_recall_curve(y_val, prob_val)
    fig_pr_v = plt.figure()
    plt.plot(rec_v, prec_v)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"VAL PR Curve (AUPRC={val_auprc:.4f})")
    wandb.log({"val_pr_curve": wandb.Image(fig_pr_v)})
    plt.close(fig_pr_v)

    prec_t, rec_t, _ = precision_recall_curve(y_test, prob_test)
    fig_pr_t = plt.figure()
    plt.plot(rec_t, prec_t)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"TEST PR Curve (AUPRC={test_auprc:.4f})")
    wandb.log({"test_pr_curve": wandb.Image(fig_pr_t)})
    plt.close(fig_pr_t)

    # -------------------------
    # (C-3) Top-k
    # -------------------------
    # "K TP 찾기 위한 조사량" 로깅 (val/test 각각)
    for k_pos in [450, 750, 1500, 3000]:
        N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(y_val.values, prob_val, k_pos)
        N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(y_test.values, prob_test, k_pos)

        wandb.log({
            # VAL
            f"val_find_{k_pos}_N_required": N_v,
            f"val_find_{k_pos}_precision": p_v,
            f"val_find_{k_pos}_recall":    r_v,
            f"val_find_{k_pos}_f1":        f_v,
            f"val_find_{k_pos}_found_tp":  found_v,

            # TEST
            f"test_find_{k_pos}_N_required": N_t,
            f"test_find_{k_pos}_precision": p_t,
            f"test_find_{k_pos}_recall":    r_t,
            f"test_find_{k_pos}_f1":        f_t,
            f"test_find_{k_pos}_found_tp":  found_t,
        })

    # per-day KPI 로깅 (VAL/TEST 각각)
    # 전제: val_df/test_df에 ts_day 컬럼 존재
    log_per_day_kpi_to_wandb("val",  val_df,  y_val.values,  prob_val,  k_list=(30,50,100,200))
    log_per_day_kpi_to_wandb("test", test_df, y_test.values, prob_test, k_list=(30,50,100,200))

    days_test = test_df["ts_day"].nunique()
    topN = 30 * days_test  # 예: 450

    # res = topN_distinct_account_metrics(
    #     df_meta=test_df, y_true=y_test.values, y_score=prob_test,
    #     topN=topN, account_col="From Account"
    # )

    res = topN_distinct_account_metrics(
        df_meta=test_df, y_true=y_test.values, y_score=prob_test,
        topN=topN, account_cols=("From Account","To Account"),  # ✅
        score_agg="max"
    )

    wandb.log({
        "test_cum_topN": res["topN_trx"],
        "test_cum_distinct_accounts": res["distinct_accounts"],
        "test_cum_tp_accounts": res["tp_accounts"],
        "test_cum_fp_accounts": res["fp_accounts"],
        "test_cum_precision_accounts": res["precision_accounts"],
    })

    dist_val  = score_bin_distribution(y_val.values,  prob_val,  n_bins=10, score_scale=1000)
    dist_test = score_bin_distribution(y_test.values, prob_test, n_bins=10, score_scale=1000)
    wandb.log({
        "val_score_bin_table":  wandb.Table(dataframe=dist_val),
        "test_score_bin_table": wandb.Table(dataframe=dist_test),
    })

    # =========================
    # (C-3b) A count-level KPI (VAL/TEST)
    # =========================

    # 1) Account-level AUC/AUPRC (기간 전체, From+To 합산)
    val_acc = aggregate_to_account_level(val_df, y_val.values, prob_val,
                                         account_cols=("From Account","To Account"),
                                         score_agg="max")
    test_acc = aggregate_to_account_level(test_df, y_test.values, prob_test,
                                          account_cols=("From Account","To Account"),
                                          score_agg="max")

    val_acc_roc, val_acc_auprc = safe_auc_metrics(val_acc["label"].values, val_acc["score"].values)
    test_acc_roc, test_acc_auprc = safe_auc_metrics(test_acc["label"].values, test_acc["score"].values)

    wandb.log({
        "val_acc_roc_auc": val_acc_roc,
        "val_acc_auprc": val_acc_auprc,
        "test_acc_roc_auc": test_acc_roc,
        "test_acc_auprc": test_acc_auprc,
        "val_n_accounts": int(len(val_acc)),
        "test_n_accounts": int(len(test_acc)),
        "val_pos_accounts": int(val_acc["label"].sum()),
        "test_pos_accounts": int(test_acc["label"].sum()),
    })

    # 2) Account-level Find-K workload (기간 전체)
    for k_pos in [450, 750, 1500, 3000]:
        N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(val_acc["label"].values, val_acc["score"].values, k_pos)
        N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(test_acc["label"].values, test_acc["score"].values, k_pos)

        wandb.log({
            f"val_acc_find_{k_pos}_N_required": N_v,
            f"val_acc_find_{k_pos}_precision": p_v,
            f"val_acc_find_{k_pos}_recall":    r_v,
            f"val_acc_find_{k_pos}_f1":        f_v,
            f"val_acc_find_{k_pos}_found_tp":  found_v,

            f"test_acc_find_{k_pos}_N_required": N_t,
            f"test_acc_find_{k_pos}_precision": p_t,
            f"test_acc_find_{k_pos}_recall":    r_t,
            f"test_acc_find_{k_pos}_f1":        f_t,
            f"test_acc_find_{k_pos}_found_tp":  found_t,
        })

    # 3) per-day account KPI (하루 단위 운영 KPI를 계좌 기준으로)
    log_per_day_account_kpi_to_wandb("val",  val_df,  y_val.values,  prob_val,  k_list=(30,50,100,200))
    log_per_day_account_kpi_to_wandb("test", test_df, y_test.values, prob_test, k_list=(30,50,100,200))

    # -------------------------
    # (C-4) threshold@recall (val에서 threshold 선택 → test에 적용)
    # -------------------------
    desired_recall = 0.90
    chosen_thr, _, _, _ = threshold_at_recall(y_val, prob_val, desired_recall)

    # ✅ thr 기반 예측(한 번만)
    y_val_pred  = (prob_val  >= chosen_thr).astype(int)
    y_test_pred = (prob_test >= chosen_thr).astype(int)

    # ✅ CM (thr 기반)
    cm_val  = confusion_matrix(y_val,  y_val_pred)
    cm_test = confusion_matrix(y_test, y_test_pred)

    fig_cm_v = plt.figure(figsize=(6,4))
    sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"VAL CM @thr={chosen_thr:.4f}")
    wandb.log({"val_confusion_matrix": wandb.Image(fig_cm_v)})
    plt.close(fig_cm_v)

    fig_cm_t = plt.figure(figsize=(6,4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"TEST CM @thr={chosen_thr:.4f}")
    wandb.log({"test_confusion_matrix": wandb.Image(fig_cm_t)})
    plt.close(fig_cm_t)

    # =========================
    # (C-4-2) Precision / Recall / F1
    # =========================

    # (A) chosen_thr 기준 (운영)
    val_precision_thr  = precision_score(y_val,  y_val_pred,  zero_division=0)
    val_recall_thr     = recall_score(y_val,     y_val_pred,  zero_division=0)
    val_f1_thr         = f1_score(y_val,         y_val_pred,  zero_division=0)

    test_precision_thr = precision_score(y_test, y_test_pred, zero_division=0)
    test_recall_thr    = recall_score(y_test,    y_test_pred, zero_division=0)
    test_f1_thr        = f1_score(y_test,        y_test_pred, zero_division=0)

    # (B) 0.5 기준 (참고)
    thr_05 = 0.5
    y_val_pred_05  = (prob_val  >= thr_05).astype(int)
    y_test_pred_05 = (prob_test >= thr_05).astype(int)

    val_precision_05  = precision_score(y_val,  y_val_pred_05,  zero_division=0)
    val_recall_05     = recall_score(y_val,     y_val_pred_05,  zero_division=0)
    val_f1_05         = f1_score(y_val,         y_val_pred_05,  zero_division=0)

    test_precision_05 = precision_score(y_test, y_test_pred_05, zero_division=0)
    test_recall_05    = recall_score(y_test,    y_test_pred_05, zero_division=0)
    test_f1_05        = f1_score(y_test,        y_test_pred_05, zero_division=0)

    # ✅ 로그(한 번에)
    wandb.log({
        "desired_recall_for_thr": float(desired_recall),
        "chosen_threshold": float(chosen_thr),

        "val_precision_thr":  float(val_precision_thr),
        "val_recall_thr":     float(val_recall_thr),
        "val_f1_thr":         float(val_f1_thr),
        "test_precision_thr": float(test_precision_thr),
        "test_recall_thr":    float(test_recall_thr),
        "test_f1_thr":        float(test_f1_thr),

        "val_precision_thr_0.5":  float(val_precision_05),
        "val_recall_thr_0.5":     float(val_recall_05),
        "val_f1_thr_0.5":         float(val_f1_05),
        "test_precision_thr_0.5": float(test_precision_05),
        "test_recall_thr_0.5":    float(test_recall_05),
        "test_f1_thr_0.5":        float(test_f1_05),
    })

    # =========================
    # (C-5) Feature Importance (gain)
    # =========================
    booster = xgb_model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")

    # (디버그) 지금 key가 뭐로 오는지 5개만 확인
    print("[DEBUG] gain_dict size:", len(gain_dict))
    print("[DEBUG] gain_dict keys sample:", list(gain_dict.keys())[:5])

    if len(gain_dict) == 0:
        print("[WARN] gain_dict is empty. (모델이 split을 거의 안 했거나, importance_type 변경 필요)")
    else:
        imp_rows = []
        for k, v in gain_dict.items():
            # 1) key가 f0,f1 형태면 feature_names로 매핑
            if isinstance(k, str) and k.startswith("f") and k[1:].isdigit():
                idx = int(k[1:])
                name = feature_names[idx] if idx < len(feature_names) else k
            else:
                # 2) key가 이미 컬럼명 형태면 그대로 사용
                name = str(k)
            imp_rows.append((name, float(v)))

        imp_rows.sort(key=lambda x: x[1], reverse=True)

        # (1) 전체 table 로깅
        table_all = wandb.Table(columns=["feature", "gain"])
        for name, gain in imp_rows:
            table_all.add_data(name, gain)
        wandb.log({"feature_importance_gain_table_all": table_all})

        # (2) bar는 top만
        topN_bar = int(getattr(config, "topn_importance", 50)) if hasattr(config, "topn_importance") else 50
        imp_top = imp_rows[:topN_bar]

        fig_imp = plt.figure(figsize=(8, 10))
        names = [x[0] for x in imp_top][::-1]
        vals  = [x[1] for x in imp_top][::-1]
        plt.barh(names, vals)
        plt.xlabel("Gain")
        plt.title(f"Top-{len(imp_top)} Feature Importance (Gain)")
        wandb.log({"feature_importance_gain_bar_top": wandb.Image(fig_imp)})
        plt.close(fig_imp)

    # =========================
    # (C-5.5) Top gain feature 기준 "test-case 영향" 테이블
    #   - 해석용이므로 X_test_raw2(가공 직후 raw)를 사용
    #   - prob_test / y_test 사용
    # =========================
    def build_feature_impact_tables(
        X_raw: pd.DataFrame,
        y_true: pd.Series,
        y_score: np.ndarray,
        top_features,
        threshold: float = 0.5,
        max_cat_levels: int = 15,
        high_value_quantile: float = 0.9,
    ):
        """
        반환:
          - binary_num_tbl: numeric/bool feature 영향 요약 (상위구간/flag)
          - cat_tbl: categorical feature 영향 요약 (카테고리별 top)
        """
        y_true = pd.Series(y_true).reset_index(drop=True)
        y_score = pd.Series(y_score).reset_index(drop=True)
        X_raw = X_raw.reset_index(drop=True)

        y_pred = (y_score >= threshold).astype(int)

        rows_num = []
        rows_cat = []

        for f in top_features:
            if f not in X_raw.columns:
                continue

            s = X_raw[f]

            # ---------- Categorical ----------
            if pd.api.types.is_categorical_dtype(s) or pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                s2 = s.astype("string").fillna("__MISSING__")

                tmp = pd.DataFrame({"x": s2, "y": y_true, "pred": y_pred})
                g = tmp.groupby("x", dropna=False).agg(
                    n=("y", "size"),
                    pos=("y", "sum"),
                    pred_pos=("pred", "sum"),
                    tp=("y", lambda v: int(((tmp.loc[v.index, "y"] == 1) & (tmp.loc[v.index, "pred"] == 1)).sum())),
                )
                # recall/precision 계산
                g["recall"] = np.where(g["pos"] > 0, g["tp"] / g["pos"], np.nan)
                g["precision"] = np.where(g["pred_pos"] > 0, g["tp"] / g["pred_pos"], np.nan)

                # 너무 희귀한 레벨은 제외(표가 너무 길어짐 방지)
                g = g.sort_values(["pos", "n"], ascending=False).head(max_cat_levels).reset_index()

                for _, r in g.iterrows():
                    rows_cat.append({
                        "feature": f,
                        "level": r["x"],
                        "n_test": int(r["n"]),
                        "pos_test": int(r["pos"]),
                        "pred_pos": int(r["pred_pos"]),
                        "tp": int(r["tp"]),
                        "recall": float(r["recall"]) if pd.notna(r["recall"]) else np.nan,
                        "precision": float(r["precision"]) if pd.notna(r["precision"]) else np.nan,
                    })
                continue

            # ---------- Numeric / Bool ----------
            # bool은 1인 케이스를 바로 보자
            if pd.api.types.is_bool_dtype(s):
                mask = s.fillna(False).astype(bool)
                cond_name = "== True"
            else:
                # 숫자형이면 상위 q 구간(기본 90%)을 “high”로 정의
                # (멘토님이 말한 "피처 케이스가 몇 건"에 가장 자연스러운 정의)
                s_num = pd.to_numeric(s, errors="coerce")
                if s_num.notna().sum() == 0:
                    continue
                qv = s_num.quantile(high_value_quantile)
                mask = s_num >= qv
                cond_name = f">= q{int(high_value_quantile*100)}({qv:.4g})"

            idx = np.where(mask.values)[0]
            if len(idx) == 0:
                continue

            y_sub = y_true.iloc[idx]
            pred_sub = y_pred.iloc[idx]

            n = len(idx)
            pos = int(y_sub.sum())
            pred_pos = int(pred_sub.sum())
            tp = int(((y_sub == 1) & (pred_sub == 1)).sum())

            recall = (tp / pos) if pos > 0 else np.nan
            precision = (tp / pred_pos) if pred_pos > 0 else np.nan

            rows_num.append({
                "feature": f,
                "condition": cond_name,
                "n_test": n,
                "pos_test": pos,
                "pred_pos": pred_pos,
                "tp": tp,
                "recall": recall,
                "precision": precision,
            })

        binary_num_tbl = pd.DataFrame(rows_num).sort_values(
            ["pos_test", "n_test"], ascending=False
        ) if len(rows_num) else pd.DataFrame(
            columns=["feature","condition","n_test","pos_test","pred_pos","tp","recall","precision"]
        )

        cat_tbl = pd.DataFrame(rows_cat).sort_values(
            ["pos_test", "n_test"], ascending=False
        ) if len(rows_cat) else pd.DataFrame(
            columns=["feature","level","n_test","pos_test","pred_pos","tp","recall","precision"]
        )

        return binary_num_tbl, cat_tbl


    # ---- top gain feature list 만들기
    topN_case = int(getattr(config, "topn_case_impact", 20)) if hasattr(config, "topn_case_impact") else 20
    top_features = [name for name, _ in imp_rows[:topN_case]]

    # ===== (1) thr = 0.5 =====
    thr_05 = 0.5
    impact_num_05, impact_cat_05 = build_feature_impact_tables(
        X_raw=X_test_raw2,
        y_true=y_test,
        y_score=prob_test,
        top_features=top_features,
        threshold=thr_05,
        max_cat_levels=15,
        high_value_quantile=0.9,
    )

    print("[DEBUG] impact_num_tbl shape:", impact_num_05.shape)
    print("[DEBUG] impact_cat_tbl shape:", impact_cat_05.shape)
    print("[DEBUG] impact_num_tbl head:\n", impact_num_05.head(5))
    print("[DEBUG] impact_cat_tbl head:\n", impact_cat_05.head(5))

    wandb.log({
        "test_case_impact_numeric_topgain_thr05": wandb.Table(dataframe=impact_num_05),
        "test_case_impact_categorical_topgain_thr05": wandb.Table(dataframe=impact_cat_05),
        "case_impact_threshold_thr05": thr_05,
    })

    # ===== (2) thr = 운영 thr (chosen_thr) =====
    thr_ops = float(chosen_thr)
    impact_num_ops, impact_cat_ops = build_feature_impact_tables(
        X_raw=X_test_raw2,
        y_true=y_test,
        y_score=prob_test,
        top_features=top_features,
        threshold=thr_ops,
        max_cat_levels=15,
        high_value_quantile=0.9,
    )

    print("[DEBUG] impact_num_tbl shape:", impact_num_ops.shape)
    print("[DEBUG] impact_cat_tbl shape:", impact_cat_ops.shape)
    print("[DEBUG] impact_num_tbl head:\n", impact_num_ops.head(5))
    print("[DEBUG] impact_cat_tbl head:\n", impact_cat_ops.head(5))

    wandb.log({
        "test_case_impact_numeric_topgain_throps": wandb.Table(dataframe=impact_num_ops),
        "test_case_impact_categorical_topgain_throps": wandb.Table(dataframe=impact_cat_ops),
        "case_impact_threshold_throps": thr_ops,
    })


    # -------------------------
    # AML 패턴 분석 로깅(4가지 케이스)
    # -------------------------

    # Val 데이터 분석
    v_50_sum, v_50_det = get_pattern_analysis_tables(val_df, prob_val, 0.5)
    v_ch_sum, v_ch_det = get_pattern_analysis_tables(val_df, prob_val, chosen_thr)

    # Test 데이터 분석 (상세 분석 포함)
    t_50_sum, t_50_det = get_pattern_analysis_tables(test_df, prob_test, 0.5)
    t_ch_sum, t_ch_det = get_pattern_analysis_tables(test_df, prob_test, chosen_thr)

    # WandB 로그 전송
    wandb.log({
        "Analysis_Val/Recall_at_0.5": v_50_sum,
        "Analysis_Val/Recall_at_Chosen": v_ch_sum,
        "Analysis_Detail/Test_Detail_at_Chosen": v_50_det,
        "Analysis_Detail/Test_Detail_at_Chosen": v_ch_det,
        "Analysis_Test/Recall_at_0.5": t_50_sum,
        "Analysis_Test/Recall_at_Chosen": t_ch_sum,
        "Analysis_Detail/Test_Detail_at_Chosen": t_50_det,
        "Analysis_Detail/Test_Detail_at_Chosen": t_ch_det # 가장 중요한 데이터
    })

    # test 데이터에 대해 3000건 적발 목표 기준 패턴 분석
    t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(test_df, prob_test, 3000)

    wandb.log({
        "Analysis_Test/Pattern_Summary_at_Find3000": t_sum_3000,
        "Analysis_Detail/Pattern_Detail_at_Find3000": t_det_3000
    })

    # 최악의 패턴 리포트 (선택 사항)
    # t_ch_sum 데이터프레임이 비어 있지 않다면 가장 낮은 Recall 패턴을 Summary에 기록 가능

    # 1. 분석에 사용할 상위 중요 피처 추출 (Top 10개)
    # 이미 코드 상단에서 계산된 imp_rows 활용
    top_features_for_analysis = [name for name, _ in imp_rows[:10]]

    # 2. 정탐/오탐/미탐 피처 및 일자별 분석 실행 (Test 데이터 기준)
    log_detailed_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=chosen_thr,
        top_features=top_features_for_analysis
    )

    log_detailed_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=0.5,
        top_features=top_features_for_analysis
    )

    # 3. 패턴 중심 에러 분석 실행
    log_pattern_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=chosen_thr
    )

    log_pattern_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=0.5
    )

    # 2. find-3000 기준으로 정탐/오탐/미탐 분석 실행 (Test 데이터셋)
    # 이 로직은 내부적으로 workload_to_find_k_positives를 호출하여 N_required를 자동으로 계산합니다.
    log_find_k_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        k_pos_target=3000,
        top_features=top_features_for_analysis
    )

    # =========================
    # (D) 콘솔 리포트 (test)
    # =========================
    print("=== Sweep Report ===")
    print("VAL  ROC-AUC:", val_roc_auc,  "VAL  AUPRC:", val_auprc)
    print("TEST ROC-AUC:", test_roc_auc, "TEST AUPRC:", test_auprc)
    print("Chosen thr(from VAL):", chosen_thr)
    print(classification_report(y_test, y_test_pred, zero_division=0))

    # # =========================
    # # (E) Artifacts 저장/업로드
    # # =========================
    # os.makedirs("artifacts_out", exist_ok=True)

    # preproc_path = "artifacts_out/transformer.joblib"
    # joblib.dump(transformer, preproc_path)

    # model_joblib_path = "artifacts_out/aml_model.joblib"
    # joblib.dump(xgb_model, model_joblib_path)

    # model_pkl_path = "artifacts_out/aml_model.pkl"
    # with open(model_pkl_path, "wb") as f:
    #     pickle.dump(xgb_model, f)

    # # (선택) 데이터 샘플 저장
    # data_sample_path = "artifacts_out/train_sample.csv"
    # try:
    #     sample_n = 2000
    #     X_train_raw_sample = X_train_raw.head(sample_n).copy()
    #     y_train_sample = y_train.loc[X_train_raw_sample.index].copy()
    #     X_train_raw_sample[LABEL_COL] = y_train_sample
    #     X_train_raw_sample.to_csv(data_sample_path, index=False)
    #     include_sample = True
    # except Exception:
    #     include_sample = False

    # art = wandb.Artifact(
    #     name="aml_bundle",
    #     type="model",
    #     description="Transformer(train-only fit) + XGB(scale_pos_weight sweep) + optional data sample"
    # )
    # art.add_file(preproc_path)
    # art.add_file(model_joblib_path)
    # art.add_file(model_pkl_path)
    # if include_sample:
    #     art.add_file(data_sample_path)

    # wandb.log_artifact(art)

    run.finish()

# 12-2) Sweep Config 정의 + 실행

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_auprc", "goal": "maximize"},  # ★ 수정
    "parameters": {
        "n_estimators": {"values": [600]},
        "max_depth": {"values": [8]},
        "learning_rate": {"values": [0.1]},
        "subsample": {"values": [0.7]},
        "colsample_bytree": {"values": [1.0]},
        "reg_lambda": {"values": [5.0]},
        "reg_alpha": {"values": [0.5]},
        "min_child_weight": {"values": [10]},
        "gamma": {"values": [0.5]},
        "scale_pos_weight": {"values": [1.0]},  # ★ 추가
    }
}

# (프로젝트/엔티티 이름은 팀 기준으로 정해줘)
sweep_id = wandb.sweep(sweep_config, project="eungyulwon")

# count: 몇 번의 실험(run)을 돌릴지
wandb.agent(sweep_id, function=train_eval_sweep, count=1)

Create sweep with ID: jwq0zt3j
Sweep URL: https://wandb.ai/0326byeol-korea-ac-kr/eungyulwon/sweeps/jwq0zt3j


wandb: Agent Starting Run: 2mwur5z9 with config:
wandb: 	colsample_bytree: 1
wandb: 	gamma: 0.5
wandb: 	learning_rate: 0.1
wandb: 	max_depth: 8
wandb: 	min_child_weight: 10
wandb: 	n_estimators: 600
wandb: 	reg_alpha: 0.5
wandb: 	reg_lambda: 5
wandb: 	scale_pos_weight: 1
wandb: 	subsample: 0.7
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


=== Raw2 categorical dtypes ===
dow_cat                    category
Receiving Currency         category
Payment Currency           category
Payment Format             category
timeofday_bucket           category
Sender_Bank_Branch_ID      category
Receiver_Bank_Branch_ID    category
Sender_Entity              category
Sender_Bank_Name           category
Receiver_Entity            category
Receiver_Bank_Name         category
From Country               category
To Country                 category
From Bank                  category
To Bank                    category
dtype: object

=== Transformed X_train info ===
type(X_train): <class 'pandas.core.frame.DataFrame'>
X_train dtype: None
[DEBUG] gain_dict size: 57
[DEBUG] gain_dict keys sample: ['Amount_Received_USD', 'Amount_Paid_USD', 'IsCrossBorder', 'IsFromFATF', 'IsToFATF']
[DEBUG] impact_num_tbl shape: (13, 8)
[DEBUG] impact_cat_tbl shape: (96, 8)
[DEBUG] impact_num_tbl head:
           feature      condition   n_test  pos_test  pred

case_impact_threshold_thr05,▁
case_impact_threshold_throps,▁
chosen_threshold,▁
desired_recall_for_thr,▁
n_train,▁
pos_test,▁
pos_train,▁
pos_val,▁
test_acc_auprc,▁
test_acc_find_1500_N_required,▁
+203,...


In [ ]:
# sweep_config = {
#     "method": "bayes",
#     "metric": {"name": "val_auprc", "goal": "maximize"},  # ★ 수정
#     "parameters": {
#         "n_estimators": {"values": [200, 400, 600]},
#         "max_depth": {"values": [3, 4, 6, 8]},
#         "learning_rate": {"values": [0.03, 0.05, 0.1]},
#         "subsample": {"values": [0.7, 0.9, 1.0]},
#         "colsample_bytree": {"values": [0.7, 0.9, 1.0]},
#         "reg_lambda": {"values": [1.0, 2.0, 5.0]},
#         "reg_alpha": {"values": [0.0, 0.1, 0.5]},
#         "min_child_weight": {"values": [1, 5, 10]},
#         "gamma": {"values": [0.0, 0.5, 1.0]},
#         "scale_pos_weight": {"values": [1, 5, 10, 20]},  # ★ 추가
#     }
# }

# # (프로젝트/엔티티 이름은 팀 기준으로 정해줘)
# sweep_id = wandb.sweep(sweep_config, project="eungyulwon")

# # count: 몇 번의 실험(run)을 돌릴지
# wandb.agent(sweep_id, function=train_eval_sweep, count=3)

# 15) (선택) Artifacts "불러오기" 예시 (필요할 때만 실행)
#     - W&B에서 best 모델 버전 지정해서 다시 로컬로 가져올 수 있음

In [ ]:
# run = wandb.init(project="aml-team-experiments", job_type="inference")
# artifact = run.use_artifact("YOUR_ENTITY/aml-team-experiments/aml_xgb_bundle:latest", type="model")
# artifact_dir = artifact.download()
# loaded_transformer = joblib.load(os.path.join(artifact_dir, "transformer.joblib"))
# loaded_model = joblib.load(os.path.join(artifact_dir, "aml_model.joblib"))
# run.finish()

# 이제 loaded_transformer/loaded_model로 동일 파이프라인 재현 가능